SQL Deep Dive - Interview Preparation Guide


# Part 1: Window Functions 

## Window Function Underlying Logic 底层逻辑

- Window functions operate on **a set of rows related to the current row**, 
- defined by `PARTITION BY` and `ORDER BY` clauses. 
- Unlike `GROUP BY` which collapses rows, window functions keep all rows while computing aggregates. 
- The execution has three phases: 
  - First, partition the data by `PARTITION BY` columns. 
  - Second, sort each partition by `ORDER BY`. 
  - Third, compute the function over the **window frame** for each row. 
    - The frame can be `ROWS` or `RANGE` based, 
    - with boundaries like `UNBOUNDED PRECEDING` to `CURRENT ROW`. 
- This allows running totals, rankings, and moving averages without self-joins.

<img src='./pic/0_Window_Functions_Cheat_Sheet-cover.png' width=850>


### Window Function 的核心组成：
```sql
function_name(column) OVER (
    PARTITION BY partition_columns    -- 分组（可选）
    ORDER BY order_columns            -- 排序（某些函数必需）
    frame_clause                      -- 窗口范围（可选）
)
```

**执行流程详解：**

```text
Step 1: PARTITION BY 分区
┌────────────────────────────────────────────┐
│ 原始数据按 PARTITION BY 列分成多个独立分区      │
│ 每个分区内部独立计算，互不影响                  │
└────────────────────────────────────────────┘
          ↓
Step 2: ORDER BY 排序
┌────────────────────────────────────────────┐
│ 每个分区内按 ORDER BY 列排序                  │
│ 这决定了窗口函数的计算顺序                     │
└────────────────────────────────────────────┘
          ↓
Step 3: Frame 计算
┌───────────────────────────────────────────┐
│ 对每一行，根据 Frame 定义确定参与计算的行集合   │
│ 在这个行集合上执行聚合/排名/分析函数            │
└───────────────────────────────────────────┘
```



### Window Frame 详解：
```sql
-- Frame 语法
ROWS BETWEEN <start> AND <end>
RANGE BETWEEN <start> AND <end>

-- 边界选项
UNBOUNDED PRECEDING   -- 分区第一行
n PRECEDING           -- 当前行往前 n 行
CURRENT ROW           -- 当前行
n FOLLOWING           -- 当前行往后 n 行
UNBOUNDED FOLLOWING   -- 分区最后一行
```

#### ROWS vs RANGE 区别：
🔹 ROWS
- 按“物理行数”计算
- 无论值是否相同，都按行数算。

🔹 RANGE
- 按“排序字段的值范围”计算
    ```sql
    RANGE BETWEEN INTERVAL '7' DAY PRECEDING AND CURRENT ROW
    -- 当前日期往前 7 天内的所有行
    -- ⚠️ 如果有相同排序值，会全部包含。
    ```
  
```sql
-- 示例数据：sales 表
-- date       | amount
-- 2024-01-01 | 100
-- 2024-01-01 | 150  -- 注意：同一天两条记录
-- 2024-01-02 | 200

-- ROWS：物理行
SELECT date, amount,
       SUM(amount) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
-- 结果：100, 250, 450（严格按行累加）

-- RANGE：逻辑范围（相同值视为同一组）
SELECT date, amount,
       SUM(amount) OVER (ORDER BY date RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
-- 结果：250, 250, 450（同日期的行一起计算）
```



### 常见窗口函数分类：

| 类型 | 函数 | 用途 |
|------|------|------|
| **排名** | ROW_NUMBER(), RANK(), DENSE_RANK(), NTILE() | 排序编号 |
| **聚合** | SUM(), AVG(), COUNT(), MAX(), MIN() | 窗口内聚合 |
| **偏移** | LAG(), LEAD(), FIRST_VALUE(), LAST_VALUE() | 访问其他行 |
| **分布** | PERCENT_RANK(), CUME_DIST() | 百分比排名 |

**实战示例：**
```sql
-- 计算每个部门的工资排名和累计工资
SELECT 
    employee_id,
    department,
    salary,
    -- 排名
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as salary_rank,
    -- 累计工资
    SUM(salary) OVER (PARTITION BY department ORDER BY salary DESC 
                      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total,
    -- 部门平均工资
    AVG(salary) OVER (PARTITION BY department) as dept_avg,
    -- 与上一个人的工资差
    salary - LAG(salary, 1, 0) OVER (PARTITION BY department ORDER BY salary DESC) as diff_from_prev
FROM employees;
```

#### LAG vs LEAD
- LAG() 用来取“当前行之前某一行”的值  
- LEAD() 用来获取“当前行之后某一行”的值
  - LEAD() 是 LAG() 的“反方向版本”。
- 不依赖 window frame
- 因为它是“行定位函数”，不是“聚合函数”。

```sql
LAG(
    column_name,    -- 要取的列
    offset,         -- 往前第几行（默认 1)
    default_value   -- 如果不存在返回什么（可选）
)
-- or
LEAD(
    column_name,    -- 要取的列
    offset,         -- 往后第几行（默认 1)
    default_value   -- 如果不存在返回什么（可选）
)
OVER (
    PARTITION BY ...
    ORDER BY ...
)
```

#### RANK vs DENSE_RANK vs ROW_NUMBER 

- All three assign rankings but handle **ties** differently. 
- `ROW_NUMBER` gives **unique sequential numbers regardless of ties** - tie values get different numbers based on physical order. 
- `RANK` handles **ties** by giving the **same rank**, then **skipping numbers** - so 1,1,3 for a tie at first place. 
- `DENSE_RANK` also gives **ties** the **same rank** but **doesn't skip** - so 1,1,2. 
- Use `ROW_NUMBER` when you need unique identifiers, `RANK` for competition-style ranking where gaps matter, and `DENSE_RANK` when you want consecutive ranks without gaps. 
- `NTILE` divides into equal groups.


**三种排名函数对比：**

```sql
-- 示例数据
-- name   | score
-- Alice  | 100
-- Bob    | 100
-- Carol  | 90
-- David  | 80

SELECT 
    name,
    score,
    ROW_NUMBER() OVER (ORDER BY score DESC) as row_num,
    RANK() OVER (ORDER BY score DESC) as rank,
    DENSE_RANK() OVER (ORDER BY score DESC) as dense_rank
FROM students;
```

| name | score | ROW_NUMBER | RANK | DENSE_RANK |
|------|-------|------------|------|------------|
| Alice | 100 | 1 | 1 | 1 |
| Bob | 100 | 2 | 1 | 1 |
| Carol | 90 | 3 | **3** | **2** |
| David | 80 | 4 | 4 | 3 |

**关键区别：**
```text
ROW_NUMBER：永远 1,2,3,4...（不管是否并列）
RANK：      并列后跳过（1,1,3,4）
DENSE_RANK：并列后不跳（1,1,2,3）
```

**使用场景：**

```sql
-- 场景1：取每组 Top 1（用 ROW_NUMBER 保证唯一）
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rn
    FROM employees
)
SELECT * FROM ranked WHERE rn = 1;

-- 场景2：比赛排名（用 RANK，第二名可能是第3名）
SELECT name, score,
       RANK() OVER (ORDER BY score DESC) as competition_rank
FROM contestants;

-- 场景3：分等级（用 DENSE_RANK，确保等级连续）
SELECT product, sales,
       DENSE_RANK() OVER (ORDER BY sales DESC) as sales_tier
FROM products;

-- 场景4：分组（用 NTILE）
SELECT customer_id, total_spend,
       NTILE(4) OVER (ORDER BY total_spend DESC) as quartile  -- 分成4组
FROM customers;
```

**NTILE 详解：**
```sql
-- NTILE(n)：将数据均分为 n 组
SELECT 
    name,
    score,
    NTILE(3) OVER (ORDER BY score DESC) as group_num
FROM students;

-- 结果：前1/3是组1，中间1/3是组2，后1/3是组3
-- 如果不能均分，前面的组会多一个
```



#### in PySpark
```python
# 1. Aggregation Window Functions
# sum(), avg(), max()/min(), count()
from pyspark.sql.window import Window
from pyspark.sql.functions import *

windowSpec = Window.partitionBy("user_id") \
                   .orderBy("event_time") \
                   .rowsBetween(-7, 0)
                   #.rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.withColumn("cum_sum", sum("amount").over(windowSpec))

# 2. Offset Functions
# lag(), lead()
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead

windowSpec = Window.partitionBy("user_id").orderBy("event_time")
df.withColumn("next_event", lead("event_time", 1).over(windowSpec))

# 3. Ranking Functions
# row_number()不跳号, rank()会跳号, dense_rank()不跳号但处理重复
windowSpec = Window.partitionBy("category").orderBy(col("sales").desc())

df.withColumn("rank", row_number().over(windowSpec)) \
  .filter("rank <= 3")

# 4. Distribution
# ntile(n)分桶, percent_rank()百分位排名, cume_dist()累积分布
```

use cases (Spark / Snowflake / BigQuery):
- 做 rolling 7 day revenue
- 做用户活跃留存曲线
- 做 session 累积
- 做 fraud 连续交易检测

## in Distributed Databases
分布式数据库中的窗口函数实现    
how **Spark SQL**, **Snowflake**, and **other distributed systems** implement window functions, including **shuffle mechanics**, **memory management**, and **optimization strategies**.   
深入探讨 Spark SQL、Snowflake 等分布式系统如何实现窗口函数，包括 Shuffle 机制、内存管理和优化策略。

### How are window functions implemented in distributed databases?
窗口函数在分布式数据库中如何实现？

- Window functions in distributed databases require **data with the same partition key** to be co-located on the **same node** for computation. 
- The implementation has three phases: 
  - first, a **shuffle or redistribution** phase moves data so rows with the same `PARTITION BY` key are on the **same node**; 
  - second, a **local sort** orders data **within each partition** by the `ORDER BY` clause; 
  - third, the **window frame computation** scans through sorted data to calculate results. 
- This is fundamentally different from single-node databases where all data is local. The partition-by clause determines data distribution, while order-by determines local sorting. 
- Without a `PARTITION BY`, all data must go to a single node — a massive scalability bottleneck.

**分布式窗口函数的核心挑战：**
- 单机数据库：所有数据在本地，直接计算
- 分布式数据库：数据分散在多个节点，必须**重新分布**才能计算

**三阶段执行模型：**

```text
阶段 1: Shuffle/Redistribute（数据重分布）
原始数据分布:
  Node 1: [A-1, B-1, C-1]
  Node 2: [A-2, B-2, C-2]
  Node 3: [A-3, B-3, C-3]

按 PARTITION BY key (A/B/C) 重分布后:
  Node 1: [A-1, A-2, A-3]  ← 所有 A 的数据
  Node 2: [B-1, B-2, B-3]  ← 所有 B 的数据
  Node 3: [C-1, C-2, C-3]  ← 所有 C 的数据
─────────────────────────────────────────
阶段 2: Local Sort（本地排序）
每个节点按 ORDER BY 列排序其分区数据
─────────────────────────────────────────
阶段 3: Window Frame Computation（窗口帧计算）
在排序后的数据上滑动窗口，计算结果
```

**关键概念：**

| 子句 | 分布式作用 | 影响 |
|------|------------|------|
| PARTITION BY | 决定数据如何**分布到节点** | 决定 Shuffle 模式 |
| ORDER BY | 决定节点内**本地排序** | 影响内存使用 |
| window frame | 决定**计算范围** | 影响计算复杂度 |

**没有 PARTITION BY 的危险：**     
```sql
-- ⚠️ 危险：所有数据必须到单个节点！
SELECT 
    ROW_NUMBER() OVER (ORDER BY created_at) as global_row_num
FROM billion_row_table;
```
后果：   
1. 单节点内存可能不够
2. 完全失去分布式并行优势
3. 可能导致 OOM (Out Of Memory) 或极慢查询

**分布式 vs 单机对比：**

| 特性 | 单机数据库 | 分布式数据库 |
|------|------------|--------------|
| 数据位置 | 本地 | 分散多节点 |
| `PARTITION BY` | 逻辑分组 | 物理数据重分布 |
| 主要开销 | 排序、内存 | **网络传输** + 排序 + 内存 |
| 无 `PARTITION BY` | 全表扫描 | **单点瓶颈** |



### How does Spark SQL shuffle data for window functions?
Spark SQL 中窗口函数如何进行 Shuffle？

- Spark SQL implements window functions through a **shuffle-sort-compute pipeline**. 
- When you use `PARTITION BY`, Spark **hash-partitions** data by those columns, shuffling rows across **executors** so **each partition key's data is co-located**. 
- Then it sorts within each partition by the `ORDER BY` columns. 
- For the actual computation, Spark uses different strategies: 
  - for **ranking functions** like `ROW_NUMBER`, it **scans once**; 
  - for **aggregate windows**, it may use a **sliding window buffer**. 
- The **shuffle is the expensive part** 
  - you can see it in the query plan as 'Exchange hashpartitioning.' 
  - If your partition keys have skew, one executor gets overloaded. 
  - **Spark 3.x AQE** can help by splitting skewed partitions, 
  - but fundamentally you need good partition key design to avoid shuffle bottlenecks.

**Spark 窗口函数执行流程：**

```text
┌─────────────────────────────────────────────────────────────────┐
│                        Spark SQL Query                          │
│  SELECT *, ROW_NUMBER() OVER (PARTITION BY dept ORDER BY sal)   │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  Stage 1: Scan + Partial Processing                             │
│  - 读取数据源（Parquet, Delta Lake 等）                            │
│  - 应用 filter/projection 下推                                   │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  Exchange (Shuffle) ← 最昂贵的操作！                              │
│  - Hash Partitioning by PARTITION BY columns                    │
│  - 数据通过网络重新分布到各 Executor                                │
│  - 相同 partition key 的数据发送到同一个 Executor                   │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  Stage 2: Sort + Window Computation                             │
│  - 按 ORDER BY 列排序（Sort）                                     │
│  - 计算窗口函数（Window）                                          │
└─────────────────────────────────────────────────────────────────┘
```

**查看执行计划：**

```python
df = spark.sql("""
    SELECT 
        department,
        employee,
        salary,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rank
    FROM employees
""")

df.explain(True)
```

**典型执行计划输出：**

```text
== Physical Plan ==
Window [row_number() windowspecdefinition(department, salary DESC, ...) AS rank]
+- Sort [department ASC, salary DESC], false, 0
   +- Exchange hashpartitioning(department, 200)   ← Shuffle！
      +- FileScan parquet [department, employee, salary]
```

**Shuffle 机制详解：**

```text
原始数据分布（3个Executor）:
┌─────────────────────────────────────────────────────────────┐
│ Executor 1        │ Executor 2        │ Executor 3         │
│ ──────────────    │ ──────────────    │ ──────────────     │
│ (Eng, Alice, 100) │ (Sales, Bob, 80)  │ (Eng, Carol, 90)   │
│ (Sales, Dave, 70) │ (Eng, Eve, 110)   │ (HR, Frank, 60)    │
│ (HR, Grace, 65)   │ (HR, Henry, 55)   │ (Sales, Ivy, 75)   │
└─────────────────────────────────────────────────────────────┘
                              │
                    Exchange hashpartitioning(department)
                              │
                              ▼
┌─────────────────────────────────────────────────────────────┐
│ Executor 1 (Eng)  │ Executor 2 (Sales) │ Executor 3 (HR)   │
│ ──────────────    │ ──────────────     │ ──────────────    │
│ (Eng, Alice, 100) │ (Sales, Bob, 80)   │ (HR, Frank, 60)   │
│ (Eng, Carol, 90)  │ (Sales, Dave, 70)  │ (HR, Grace, 65)   │
│ (Eng, Eve, 110)   │ (Sales, Ivy, 75)   │ (HR, Henry, 55)   │
└─────────────────────────────────────────────────────────────┘
```

**Spark 窗口函数的内部实现类型：**

| 函数类型 | 实现策略 | 内存使用 |
|----------|----------|----------|
| ROW_NUMBER, RANK, DENSE_RANK | 单次扫描计数 | 低 |
| LAG, LEAD | 缓冲 N 行 | 低（固定大小） |
| SUM, AVG (UNBOUNDED) | 累积计算 | 中 |
| SUM, AVG (滑动窗口) | 滑动缓冲区 | 取决于窗口大小 |
| FIRST_VALUE, LAST_VALUE | 可能需要完整分区 | 高（无界时） |

**多个窗口函数的优化：**

```python
# Spark 会合并相同 PARTITION BY + ORDER BY 的窗口
df = spark.sql("""
    SELECT 
        department,
        salary,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rn,
        RANK()       OVER (PARTITION BY department ORDER BY salary DESC) as rnk,
        SUM(salary)  OVER (PARTITION BY department ORDER BY salary DESC) as running_sum
    FROM employees
""")

# 执行计划中只有一次 Shuffle + Sort！
# Window [row_number(), rank(), sum(salary)]
# +- Sort [department, salary DESC]
#    +- Exchange hashpartitioning(department)
```

**不同 PARTITION BY 会导致多次 Shuffle：**

```python
# ⚠️ 两个不同的 PARTITION BY = 两次 Shuffle！
df = spark.sql("""
    SELECT 
        department,
        region,
        salary,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary) as dept_rank,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY salary) as region_rank
    FROM employees
""")

# 执行计划会有两次 Exchange！
```
- Spark 的 window 需要根据 PARTITION BY 做 hash partition。
- 不同 partition key 会触发独立 shuffle。
- 优化方式包括：
    - 拆分查询 + cache
    - 预 repartition
    - 物化中间结果
    - 业务允许的话减少窗口数量

如果数据量：
- < 10GB
- 或 partition key 基数很高
- 或 executor 内存足够

两次 shuffle 其实没问题。

#### Optimization
对上述的两次shuffle优化方法     
**✅ 方案 1：拆成两个 DataFrame + Cache（最实用**   
```python
df1 = spark.sql("""
    SELECT 
        department,
        region,
        salary,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary) as dept_rank
    FROM employees
""").cache()

df2 = spark.sql("""
    SELECT 
        department,
        region,
        salary,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY salary) as region_rank
    FROM employees
""")

result = df1.join(
    df2.select("department", "region", "salary", "region_rank"),
    ["department", "region", "salary"]
)
```
优点：
- 每个 DF 只 shuffle 一次
- 可以缓存复用

缺点：
- 需要 join


**✅ 方案 2：预分区（如果某个 key 使用更多）**        

如果 80% 查询都是按 department：
```python
df = employees.repartition("department")
```
优点：
- department 的 window 不再 shuffle
- region 的还是会 shuffle
- 减少一半成本

**✅ 方案 3：物理分层计算（推荐在大数据场景）**   
```text
step 1: 生成 dept_rank → 写 parquet
step 2: 读 parquet → 生成 region_rank
```
适合：
- TB 级数据
- 离线批处理

### How does Snowflake optimize window functions?
Snowflake 如何优化窗口函数？

Snowflake optimizes window functions through several mechanisms:  
- First, its **micro-partition architecture** means data is already distributed, and Snowflake's **query optimizer** can **minimize data movement** when partition keys align with micro-partition clustering. 
- Second, it uses a technique called **'window function pipelining'** — multiple window functions with the same partitioning are computed in a single pass. 
- Third, Snowflake automatically spills to its **remote disk layer** when memory is insufficient, leveraging its separation of compute and storage. 
- Fourth, for common patterns like **'qualify row_number() = 1'**, Snowflake has specific optimizations to avoid computing unnecessary rows. 
- The query profile shows 'WindowFunction' operators with metrics for memory usage and spilling — always check these for large window operations.


**Snowflake 架构优势：**

```text
┌─────────────────────────────────────────────────────────────┐
│                    Snowflake 架构                            │
├─────────────────────────────────────────────────────────────┤
│  Cloud Services Layer（查询优化、元数据）                      │
├─────────────────────────────────────────────────────────────┤
│  Virtual Warehouse（计算层，可独立扩展）                       │
│    ├── Warehouse 1 (XS)                                     │
│    ├── Warehouse 2 (L)                                      │
│    └── Warehouse 3 (XL)                                     │
├─────────────────────────────────────────────────────────────┤
│  Storage Layer（S3/Azure Blob/GCS，独立于计算）               │
│    └── Micro-partitions（每个 50-500MB，自动压缩）           │
└─────────────────────────────────────────────────────────────┘
```

**Snowflake 窗口函数优化策略：**

**1. Micro-partition 裁剪（Pruning）**

```sql
-- 如果表按 order_date 聚簇
-- 且窗口函数按 order_date 分区
-- Snowflake 可以减少数据扫描

SELECT 
    order_date,
    customer_id,
    amount,
    SUM(amount) OVER (PARTITION BY order_date ORDER BY customer_id) as running_total
FROM orders
WHERE order_date BETWEEN '2024-01-01' AND '2024-01-31';

-- Micro-partition pruning 会先过滤不相关的分区
```

**2. 窗口函数流水线（Pipelining）**

```sql
-- 相同 PARTITION BY + ORDER BY 的窗口函数合并计算
SELECT 
    department,
    employee,
    salary,
    ROW_NUMBER() OVER w as rn,
    RANK()       OVER w as rnk,
    DENSE_RANK() OVER w as drnk,
    SUM(salary)  OVER w as running_sum,
    AVG(salary)  OVER w as running_avg
FROM employees
WINDOW w AS (PARTITION BY department ORDER BY salary DESC);

-- Snowflake 内部：单次数据传递，计算所有窗口函数
```

**3. QUALIFY 优化（Snowflake 特有语法）**

```sql
-- 传统写法：先计算所有行，再过滤
SELECT * FROM (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) as rn
    FROM orders
)
WHERE rn = 1;

-- Snowflake QUALIFY：可以提前终止计算
SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) as rn
FROM orders
QUALIFY rn = 1;  -- Snowflake 特有语法

-- 优化效果：对于 rn = 1，找到第一行后可能跳过后续计算
```

**4. 自动 Spill to Disk**

```sql
-- Snowflake 自动处理内存不足的情况
-- 不需要手动配置，但可以通过 Query Profile 监控

-- 在 Query Profile 中查看：
-- WindowFunction 算子的 "Bytes spilled to local disk"
-- WindowFunction 算子的 "Bytes spilled to remote disk"
```

**Snowflake Query Profile 分析：**

```text
┌─────────────────────────────────────────────────────────────┐
│  Query Profile - WindowFunction Operator                    │
├─────────────────────────────────────────────────────────────┤
│  Statistics:                                                │
│    - Input rows: 100,000,000                                │
│    - Output rows: 100,000,000                               │
│    - Memory used: 2.5 GB                                    │
│    - Bytes spilled to local: 500 MB    ← 溢出到本地SSD      │
│    - Bytes spilled to remote: 0        ← 溢出到远程存储     │
│    - Partitions processed: 1,000                            │
│    - Partitions with spilling: 50      ← 50个分区发生溢出   │
└─────────────────────────────────────────────────────────────┘

关注指标：
- spilled to local: 可接受，使用本地 SSD，影响较小
- spilled to remote: 警告信号，使用远程存储，显著影响性能
```

**Snowflake vs Spark 窗口函数对比：**

| 特性 | Snowflake | Spark |
|------|-----------|-------|
| 数据分布 | 自动管理 micro-partitions | 用户控制分区 |
| 内存管理 | 自动 spill，透明 | 需要配置 |
| QUALIFY 语法 | ✅ 支持 | ❌ 不支持 |
| 查询分析 | Query Profile GUI | Spark UI + explain() |
| 扩展方式 | 调整 Warehouse 大小 | 增加 Executor |



### Do window functions spill to disk? When and how?
窗口函数会 Spill to Disk 吗？何时以及如何发生？

- Yes, window functions can and do spill to disk **when memory is insufficient**. 
- In **Spark**
  - this happens when the data for **a single partition** exceeds the available execution memory 
  - Spark uses an **external sort algorithm** that spills sorted runs to disk. 
  - The spillage occurs during both **the sort phase** and potentially during **window computation** if the buffer exceeds memory. 
- In **Snowflake**
  - spilling happens automatically with two tiers: first to local SSD on the compute node, then to remote storage if local disk fills up. 
- You can detect spilling in Spark through the **Spark UI's stage details** showing 'spill (memory)' and 'spill (disk)' metrics, 
- or in Snowflake through **the Query Profile's 'Bytes spilled' statistics**. 
- Excessive spilling indicates you need either more memory, better partitioning, or query optimization.

**Spill to Disk 的触发条件：**

```text
┌───────────────────────────────────────────────────────────┐
│                    内存使用情况                             │
├───────────────────────────────────────────────────────────┤
│                                                           │
│  ████████████████████░░░░░░░░░░░░░░░░░░░  正常：内存足够    │
│  ████████████████████████████████░░░░░░░  警告：接近上限    │
│  ████████████████████████████████████████  溢出：Spill!    │
│                                          │                │
│                                    Memory Threshold       │
└───────────────────────────────────────────────────────────┘

当单个分区的数据量超过可用内存时，发生 Spill
```

**Spark 中的 Spill 机制：**

```python
# Spark 内存模型
# spark.executor.memory = 8g 时的内存分配

"""
┌─────────────────────────────────────────────────────────────┐
│  Executor Memory (8 GB)                                     │
├─────────────────────────────────────────────────────────────┤
│  Reserved Memory (300 MB fixed)                             │
├─────────────────────────────────────────────────────────────┤
│  User Memory (40%): 存储用户数据结构                           │
│  Unified Memory (60%):                                      │
│    ├── Storage Memory: 缓存 RDD/DataFrame                    │
│    └── Execution Memory: Shuffle, Sort, Window              │
│              ↑                                              │
│        窗口函数使用这部分内存                                   │
└─────────────────────────────────────────────────────────────┘
"""

# 相关配置
spark.conf.set("spark.executor.memory", "8g")
spark.conf.set("spark.memory.fraction", "0.6")      # Unified Memory 占比
spark.conf.set("spark.memory.storageFraction", "0.5") # Storage 初始占比
```

**Spark Spill 过程详解：**

```text
窗口函数执行时：

1. Sort 阶段（排序）
   ┌──────────────────────────────────────────────────────────┐
   │  数据块 → 内存排序 → 如果内存满 → 写入磁盘（sorted run）   │
   │                                                          │
   │  [Unsorted Data] → [Memory Buffer] → [Disk: sorted_run_1]│
   │                  → [Memory Buffer] → [Disk: sorted_run_2]│
   │                  → ...                                   │
   │                                                          │
   │  最后：合并所有 sorted runs（归并排序）                    │
   └──────────────────────────────────────────────────────────┘

2. Window 阶段（窗口计算）
   ┌──────────────────────────────────────────────────────────┐
   │  对于 UNBOUNDED PRECEDING 类型的窗口：                    │
   │  - 可能需要缓存整个分区                                   │
   │  - 如果分区太大 → Spill 窗口缓冲区                        │
   │                                                          │
   │  对于 ROWS BETWEEN N PRECEDING AND CURRENT ROW：          │
   │  - 只需缓存 N+1 行                                        │
   │  - 很少发生 Spill                                        │
   └──────────────────────────────────────────────────────────┘
```

**在 Spark UI 中查看 Spill：**

```text
┌─────────────────────────────────────────────────────────────┐
│  Stage 2: Window                                            │
├─────────────────────────────────────────────────────────────┤
│  Task ID │ Duration │ Spill (Memory) │ Spill (Disk)        │
│  ────────┼──────────┼────────────────┼─────────────────    │
│  0       │ 45s      │ 2.5 GB         │ 1.8 GB              │
│  1       │ 12s      │ 0              │ 0                   │
│  2       │ 48s      │ 2.8 GB         │ 2.1 GB              │
│  3       │ 10s      │ 0              │ 0                   │
└─────────────────────────────────────────────────────────────┘

分析：
- Task 0 和 2 发生了严重的 Spill → 可能是数据倾斜
- Spill (Memory): 从执行内存移出的数据量
- Spill (Disk): 实际写入磁盘的数据量（压缩后）
```

**Snowflake Spill 的两级结构：**

```text
┌─────────────────────────────────────────────────────────────┐
│                  Snowflake Spill 层级                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Level 1: Memory                                            │
│     ↓ 内存不足                                               │
│  Level 2: Local SSD (Compute Node)   ← 快，影响小            │
│     ↓ 本地磁盘不足                                           │
│  Level 3: Remote Storage (S3/Blob)   ← 慢，影响大            │
│                                                             │
└─────────────────────────────────────────────────────────────┘

Query Profile 中的指标：
- "Bytes spilled to local storage": Level 2
- "Bytes spilled to remote storage": Level 3
```

**Spill 对性能的影响：**

| Spill 类型 | 延迟影响 | 原因 |
|------------|----------|------|
| 无 Spill | 基准 | 纯内存操作 |
| Spill to Local SSD | 2-5x 慢 | SSD I/O |
| Spill to Remote (Spark) | 10-50x 慢 | 网络 + 磁盘 |
| Spill to Remote (Snowflake) | 5-20x 慢 | S3/Blob I/O |


### Question 5: How to avoid massive memory usage in window functions?
如何避免窗口函数的大量内存使用？

> "Several strategies help control memory usage. First and most important: always use PARTITION BY to distribute work — without it, all data goes to one node. Second, choose appropriate partition keys with reasonable cardinality — too few partitions means each is too large, too many means overhead. Third, avoid UNBOUNDED window frames when possible — use fixed-size frames like 'ROWS BETWEEN 100 PRECEDING AND CURRENT ROW' instead of 'UNBOUNDED PRECEDING.' Fourth, filter data before the window function, not after — this reduces the dataset being windowed. Fifth, break up queries with multiple different partition keys into separate stages. Sixth, in Spark, tune shuffle partitions and executor memory. In Snowflake, use larger warehouse sizes. Monitor spilling metrics and adjust accordingly."

**策略 1：始终使用 PARTITION BY（最重要！）**

```sql
-- ❌ 极差：所有数据到单节点
SELECT 
    ROW_NUMBER() OVER (ORDER BY created_at) as global_rank
FROM events;  -- 10 亿行全部到一个节点 → OOM

-- ✅ 好：数据分散到多个节点
SELECT 
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at) as user_rank
FROM events;  -- 每个 user_id 独立处理

-- ✅ 更好：如果不需要精确全局排名，用近似方法
SELECT 
    date_trunc('day', created_at) as day,
    ROW_NUMBER() OVER (PARTITION BY date_trunc('day', created_at) ORDER BY created_at) as daily_rank
FROM events;
```

**策略 2：选择合适的分区键粒度**

```sql
-- ❌ 分区太少（每个分区太大）
SELECT ... OVER (PARTITION BY country ORDER BY ...)  -- 可能只有 ~200 个分区

-- ❌ 分区太多（调度开销大）
SELECT ... OVER (PARTITION BY user_id, session_id, event_id ORDER BY ...)  -- 可能数十亿分区

-- ✅ 合适的粒度
SELECT ... OVER (PARTITION BY user_id ORDER BY ...)  -- 通常合适

-- 或者组合键达到合适粒度
SELECT ... OVER (PARTITION BY country, date_trunc('month', order_date) ORDER BY ...)
```

**分区数量经验法则：**

| 数据量 | 推荐分区数 | 每分区大小 |
|--------|-----------|-----------|
| 1 GB | 100-200 | 5-10 MB |
| 100 GB | 1,000-2,000 | 50-100 MB |
| 1 TB | 5,000-10,000 | 100-200 MB |
| 10 TB | 20,000-50,000 | 200-500 MB |

**策略 3：避免 UNBOUNDED 窗口帧**

```sql
-- ❌ UNBOUNDED：需要缓存整个分区
SELECT 
    SUM(amount) OVER (
        PARTITION BY customer_id 
        ORDER BY order_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as running_total
FROM orders;

-- ✅ 固定窗口：只需要缓存固定行数
SELECT 
    AVG(amount) OVER (
        PARTITION BY customer_id 
        ORDER BY order_date 
        ROWS BETWEEN 30 PRECEDING AND CURRENT ROW  -- 只缓存 31 行
    ) as moving_avg_30d
FROM orders;

-- ✅ 如果必须用 UNBOUNDED，考虑预聚合
-- 先按天/周聚合，减少数据量，再计算窗口
WITH daily_totals AS (
    SELECT customer_id, date, SUM(amount) as daily_amount
    FROM orders
    GROUP BY customer_id, date
)
SELECT 
    SUM(daily_amount) OVER (PARTITION BY customer_id ORDER BY date) as running_total
FROM daily_totals;
```

**策略 4：先过滤，后窗口**

```sql
-- ❌ 在所有数据上计算窗口，然后过滤
SELECT * FROM (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) as rn
    FROM orders  -- 1亿行
)
WHERE order_date >= '2024-01-01'  -- 过滤太晚
  AND rn = 1;

-- ✅ 先过滤，减少窗口计算的数据量
SELECT * FROM (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) as rn
    FROM orders
    WHERE order_date >= '2024-01-01'  -- 先过滤！1000万行
)
WHERE rn = 1;
```

**策略 5：拆分不同 PARTITION BY 的查询**

```python
# ❌ 单个查询中多个不同的 PARTITION BY
df = spark.sql("""
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY date) as cust_rank,
        ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY date) as prod_rank,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY date) as region_rank
    FROM sales
""")

# ✅ 分步计算，可以更好地控制资源
df1 = spark.sql("""
    SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY date) as cust_rank
    FROM sales
""")
df1.cache()  # 缓存中间结果

df2 = spark.sql("""
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY date) as prod_rank
    FROM cached_df1
""")
# ...
```

**策略 6：Spark 配置调优**

```python
# 增加 Executor 内存
spark.conf.set("spark.executor.memory", "16g")
spark.conf.set("spark.executor.memoryOverhead", "4g")  # Off-heap 内存

# 增加 Shuffle 分区数（减少每个分区大小）
spark.conf.set("spark.sql.shuffle.partitions", "2000")  # 默认 200

# 或使用 AQE 自动调整
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# 增加排序内存
spark.conf.set("spark.sql.windowExec.buffer.spill.threshold", "4096")  # 默认 4096 行
```

**策略 7：Snowflake 优化**

```sql
-- 使用更大的 Warehouse
ALTER WAREHOUSE my_warehouse SET WAREHOUSE_SIZE = 'XLARGE';

-- 使用 QUALIFY 提前终止
SELECT *
FROM orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) = 1;

-- 检查聚簇键是否与 PARTITION BY 对齐
ALTER TABLE orders CLUSTER BY (customer_id);  -- 如果经常按 customer_id 分区
```

**内存优化检查清单：**

| 检查项 | 操作 |
|--------|------|
| 有 PARTITION BY 吗？ | 必须有，除非数据很小 |
| 分区数合适吗？ | 数千到数万，每个 100-500 MB |
| 用了 UNBOUNDED 吗？ | 尽量改为固定窗口 |
| 先过滤了吗？ | WHERE 应该在窗口之前 |
| 有多个不同 PARTITION BY 吗？ | 考虑拆分查询 |
| 监控 Spill 了吗？ | 检查 Spark UI / Snowflake Profile |



### Question 6: What happens with data skew in window functions?

窗口函数遇到数据倾斜会怎样？


> "Data skew in window functions is particularly dangerous because one partition with disproportionately more data than others creates a single overloaded executor. While other executors finish quickly, this 'straggler' becomes the bottleneck. In Spark, you'll see one task taking 10-100x longer than others in the stage. The skewed partition may spill to disk or OOM. Solutions include: salting the partition key to split hot partitions artificially, using Spark 3.x AQE which can automatically split skewed partitions, pre-filtering known hot keys for separate processing, or adding a secondary partition dimension to spread the load. For window functions specifically, consider whether you really need global ranking across skewed keys or if approximate methods would suffice."


**数据倾斜的表现：**

```text
正常情况（均匀分布）:
┌────────────────────────────────────────────────────────────┐
│  Task 1: ████████  (10 sec)                                │
│  Task 2: ████████  (10 sec)                                │
│  Task 3: ████████  (10 sec)                                │
│  Task 4: ████████  (10 sec)                                │
│  总耗时: 10 秒                                              │
└────────────────────────────────────────────────────────────┘

数据倾斜（某个 partition key 数据量特别大）:
┌────────────────────────────────────────────────────────────┐
│  Task 1: ██  (2 sec)                                       │
│  Task 2: ██  (2 sec)                                       │
│  Task 3: ██  (2 sec)                                       │
│  Task 4: ████████████████████████████████████  (100 sec)  │ ← 拖累整体！
│  总耗时: 100 秒                                             │
└────────────────────────────────────────────────────────────┘
```

**窗口函数中倾斜的典型场景：**

```sql
-- 场景：电商订单按商家分区
-- 问题：大商家（如天猫超市）有数亿订单，小商家只有几十个

SELECT 
    merchant_id,
    order_id,
    amount,
    ROW_NUMBER() OVER (PARTITION BY merchant_id ORDER BY order_time) as order_rank
FROM orders;

-- merchant_id = 'tmall_supermarket' 的分区可能有 1 亿行
-- merchant_id = 'small_shop_123' 的分区可能只有 50 行
-- 处理 tmall 的 Task 会成为瓶颈
```

**Spark UI 中识别倾斜：**

```text
┌─────────────────────────────────────────────────────────────┐
│  Stage 2: Window (4 tasks)                                  │
├─────────────────────────────────────────────────────────────┤
│  Summary Metrics:                                           │
│    Duration:                                                │
│      Min: 2 sec                                             │
│      25th percentile: 3 sec                                 │
│      Median: 4 sec                                          │
│      75th percentile: 5 sec                                 │
│      Max: 180 sec  ← 最大值远超中位数 = 严重倾斜！            │
│                                                             │
│    Shuffle Read:                                            │
│      Min: 10 MB                                             │
│      Max: 15 GB   ← 最大值远超其他 = 数据倾斜               │
└─────────────────────────────────────────────────────────────┘
```

**解决方案 1：Key Salting（加盐）**

```python
# 对于热点 partition key，添加随机后缀分散到多个分区

from pyspark.sql.functions import col, concat, lit, floor, rand, collect_list

# 原始查询（有倾斜）
df.sql("""
    SELECT merchant_id, order_id,
           ROW_NUMBER() OVER (PARTITION BY merchant_id ORDER BY order_time) as rn
    FROM orders
""")

# 加盐处理（两步走）
SALT_BUCKETS = 10

# Step 1: 添加盐值，分散计算
salted_df = spark.sql(f"""
    SELECT 
        merchant_id,
        order_id,
        order_time,
        concat(merchant_id, '_', floor(rand() * {SALT_BUCKETS})) as salted_key,
        floor(rand() * {SALT_BUCKETS}) as salt
    FROM orders
""")

# Step 2: 在 salted_key 上计算窗口（分散负载）
windowed_df = salted_df.sql("""
    SELECT 
        merchant_id,
        order_id,
        ROW_NUMBER() OVER (PARTITION BY salted_key ORDER BY order_time) as local_rank,
        salt
    FROM salted_df
""")

# Step 3: 如果需要全局排名，需要额外合并步骤
# 这取决于业务需求是否真的需要精确全局排名
```

**解决方案 2：Spark AQE 自动处理倾斜**

```python
# Spark 3.x AQE 可以自动检测和拆分倾斜分区
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# 针对 Window 的倾斜优化（Spark 3.2+）
spark.conf.set("spark.sql.adaptive.optimizeSkewsInRebalancePartitions.enabled", "true")

# 调整倾斜检测阈值
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256m")
```

**解决方案 3：分离热点 Key 单独处理**

```python
# 如果知道哪些 key 是热点
HOT_MERCHANTS = ['tmall_supermarket', 'jd_self', 'pdd_flagship']

# 热点数据单独处理
hot_df = spark.sql(f"""
    SELECT 
        merchant_id,
        order_id,
        ROW_NUMBER() OVER (PARTITION BY merchant_id, date_trunc('hour', order_time) 
                          ORDER BY order_time) as hourly_rank
    FROM orders
    WHERE merchant_id IN {tuple(HOT_MERCHANTS)}
""")

# 非热点数据正常处理
normal_df = spark.sql(f"""
    SELECT 
        merchant_id,
        order_id,
        ROW_NUMBER() OVER (PARTITION BY merchant_id ORDER BY order_time) as rank
    FROM orders
    WHERE merchant_id NOT IN {tuple(HOT_MERCHANTS)}
""")

# 合并结果
result = hot_df.union(normal_df)
```

**解决方案 4：增加分区维度**

```sql
-- 原始：单一维度分区（容易倾斜）
SELECT 
    merchant_id,
    ROW_NUMBER() OVER (PARTITION BY merchant_id ORDER BY order_time) as rn
FROM orders;

-- 优化：增加时间维度，分散每个商家的数据
SELECT 
    merchant_id,
    date_trunc('day', order_time) as order_day,
    ROW_NUMBER() OVER (
        PARTITION BY merchant_id, date_trunc('day', order_time) 
        ORDER BY order_time
    ) as daily_rank
FROM orders;

-- 效果：tmall 一年的 1 亿订单 → 分散到 365 个分区，每个约 27 万
```

**倾斜处理决策树：**

```text
发现窗口函数执行慢
        │
        ▼
检查是否数据倾斜？（Spark UI 查看 Task 耗时差异）
        │
   ┌────┴────┐
   │         │
  否        是
   │         │
   ▼         ▼
检查其他     确定热点 Key
问题         │
        ┌────┴────┐
        │         │
    已知热点    未知热点
        │         │
        ▼         ▼
   分离处理    使用 AQE 或
   或加盐      增加分区维度
```


### Summary Table

| 主题 | 关键点 | Spark | Snowflake |
|------|--------|-------|-----------|
| **分布式实现** | Shuffle + Sort + Compute | Exchange + Sort + Window | 自动管理 |
| **Shuffle 触发** | PARTITION BY 决定 | hashpartitioning | 内部优化 |
| **内存管理** | 需要手动配置 | executor.memory | 自动 spill |
| **Spill 检测** | Spark UI | UI 指标 | Query Profile |
| **倾斜处理** | AQE / 手动加盐 | AQE / 手动 | 自动 + 更大 Warehouse |
| **优化语法** | N/A | N/A | QUALIFY |
| **最佳实践** | 始终用 PARTITION BY | 同左 | 同左 |
| **避免 UNBOUNDED** | 用固定窗口帧 | 同左 | 同左 |
| **先过滤后窗口** | WHERE 在窗口前 | 同左 | 同左 |



### Quick Reference: Memory Optimization | 快速参考：内存优化

```sql
-- ❌ 避免
ROW_NUMBER() OVER (ORDER BY col)                    -- 无 PARTITION BY
SUM(x) OVER (PARTITION BY a ROWS UNBOUNDED PRECEDING) -- UNBOUNDED
SELECT * FROM (SELECT ... WINDOW ...) WHERE filter  -- 后过滤

-- ✅ 推荐
ROW_NUMBER() OVER (PARTITION BY key ORDER BY col)   -- 有 PARTITION BY
AVG(x) OVER (PARTITION BY a ROWS 30 PRECEDING)      -- 固定窗口
SELECT ... WINDOW ... FROM (SELECT ... WHERE filter) -- 先过滤
```

**Spark 配置快速参考：**

```python
spark.conf.set("spark.executor.memory", "16g")
spark.conf.set("spark.sql.shuffle.partitions", "2000")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---


# Part 2: GROUP BY & Aggregation | 分组与聚合


## GROUP BY Execution Flow 执行流程

`GROUP BY` execution follows these steps:  
- First, the `FROM` clause reads data and `WHERE` filters rows. 
- Second, remaining rows are partitioned by `GROUP BY` columns - this often requires a **sort or hash operation**. 
- Third, aggregate functions compute over each group. 
- Fourth, `HAVING` filters groups based on aggregate results. 
- Finally, `SELECT` projects the output columns. 

Under the hood, databases use either 
- **hash** aggregation - building a hash table with group keys 
- or **sort-based** aggregation - sorting data then processing consecutive rows. 
- Hash is faster for distinct groups, sort is better when data is already ordered."


**执行顺序（逻辑顺序）：**
```sql
SELECT department, COUNT(*), AVG(salary)    -- 5. 投影
FROM employees                               -- 1. 读取数据
WHERE status = 'active'                      -- 2. 行过滤
GROUP BY department                          -- 3. 分组
HAVING COUNT(*) > 5                          -- 4. 组过滤
ORDER BY AVG(salary) DESC;                   -- 6. 排序
```

```text
执行流程图：

FROM employees
      ↓
WHERE status = 'active'    ← 行级过滤（减少数据量）
      ↓
GROUP BY department        ← 分组操作
      ↓
  ┌───────────────────────────┐
  │  聚合计算                  │
  │  - COUNT(*)               │
  │  - AVG(salary)            │
  │  每个 group 产生一行结果     │
  └───────────────────────────┘
      ↓
HAVING COUNT(*) > 5        ← 组级过滤
      ↓
SELECT / ORDER BY          ← 投影和排序
```

### **两种聚合实现方式：**

#### Hash Aggregation（哈希聚合）

原理： 
1. 创建 Hash Table，key = GROUP BY 列     
2. 遍历数据，每行计算 hash(group_key)      
3. 在 hash table 中找到对应桶，更新聚合值     
4. 遍历完成后，输出 hash table 中所有条目      

- 优点：O(n) 时间复杂度，不需要排序
- 缺点：需要足够内存存储 hash table
- 适用：GROUP BY 列基数（distinct 值）不太大


#### Sort-Based Aggregation（排序聚合）

原理：
1. 按 GROUP BY 列排序    
2. 顺序扫描，相同 key 的行连续出现       
3. 遇到新 key 时，输出上一组的聚合结果       

- 优点：内存使用可控，适合大基数
- 缺点：需要排序，O(n log n)
- 适用：数据已排序，或 GROUP BY 列基数很大


#### 执行计划中的体现 (EXPLAIN)
```sql
EXPLAIN SELECT department, COUNT(*) 
FROM employees 
GROUP BY department;

-- 可能看到：
-- HashAggregate (Hash 聚合)
-- 或
-- Sort → GroupAggregate (排序聚合)
```

### 优化技巧：
```sql
-- 1. 先 WHERE 过滤，减少参与 GROUP BY 的数据量
SELECT department, COUNT(*)
FROM employees
WHERE hire_date > '2020-01-01'  -- 先过滤
GROUP BY department;

-- 2. 避免 SELECT *，只选需要的列
SELECT department, COUNT(*)  -- 不要 SELECT *
FROM employees
GROUP BY department;

-- 3. 如果只需要部分分组，用 HAVING 或提前过滤
SELECT department, COUNT(*)
FROM employees
WHERE department IN ('Sales', 'Engineering')  -- 提前过滤比 HAVING 好
GROUP BY department;
```


## COUNT(DISTINCT) Optimization

**COUNT DISTINCT** is expensive because it requires **tracking all unique values**.   

Optimization strategies include: 
- First, use approximate algorithms like **HyperLogLog** when exact count isn't needed - gives **97% accuracy with minimal memory**. 
- Second, **pre-aggregate in a subquery or CTE** to reduce data before the distinct count. 
- Third, for multiple COUNT DISTINCTs, consider **GROUP BY with CASE statements** to avoid multiple passes. 
- Fourth, in **distributed systems** like Spark, **two-stage aggregation** (local distinct then global distinct ) to reduces shuffle. 
- Finally, **bitmap indexes** can accelerate distinct counts on low-cardinality columns.

**COUNT(DISTINCT) 为什么慢？**

需要记录所有不同的值才能计数
- 内存：需要存储所有 distinct 值
- CPU：每个值都要检查是否已存在
- 分布式场景：需要全局去重，shuffle 数据量大


**优化策略：**

### 策略1：近似算法 (HyperLogLog)
```sql
-- 标准 SQL（精确）
SELECT COUNT(DISTINCT user_id) FROM events;  -- 慢

-- Spark SQL（近似，误差约 2%）
SELECT APPROX_COUNT_DISTINCT(user_id) FROM events;  -- 快很多

-- PostgreSQL
SELECT COUNT(DISTINCT user_id) FROM events;  -- 可用 HyperLogLog 扩展，默认不带，需要安装，常用：postgresql-hll（由 Citus 提供）

-- BigQuery
SELECT APPROX_COUNT_DISTINCT(user_id) FROM events;
```

**HyperLogLog 原理简述：**

核心思想：利用哈希值的统计特性估算基数     
- 将元素 hash 成二进制
- 记录最长的前导零序列
- 通过概率公式估算不同元素数量
- 内存极小（几KB）可估算数十亿基数
- 误差约 2%，可接受时优先使用


### 策略2：预聚合 + 二次聚合
```sql
-- 原始查询（一次性处理大量数据）
SELECT date, COUNT(DISTINCT user_id)
FROM events
GROUP BY date;

-- 优化：先在子查询中去重，再计数
SELECT date, COUNT(user_id)
FROM (
    SELECT DISTINCT date, user_id
    FROM events
) t
GROUP BY date;

-- 或使用 CTE
WITH distinct_users AS (
    SELECT DISTINCT date, user_id FROM events
)
SELECT date, COUNT(user_id)
FROM distinct_users
GROUP BY date;
```

### 策略3：多个 COUNT(DISTINCT) 优化
```sql
-- 不好：多次扫描
SELECT 
    COUNT(DISTINCT user_id),
    COUNT(DISTINCT session_id),
    COUNT(DISTINCT product_id)
FROM events;

-- 优化：使用 GROUP BY + 条件聚合（单次扫描）
SELECT 
    COUNT(DISTINCT user_id) as users,
    COUNT(DISTINCT session_id) as sessions,
    COUNT(DISTINCT product_id) as products
FROM events;
-- 某些数据库会优化，但如果不行，考虑分开查询后 JOIN

-- Spark 特有优化：expand + 两阶段聚合
-- 通过 expand 操作将多个 distinct 转换为单列处理
```

### 策略4：Spark 两阶段聚合
```sql
-- Spark 内部优化原理
-- 阶段1：本地去重（每个 partition 内）
-- 阶段2：全局去重（shuffle 后）

-- 手动实现类似效果
-- Step 1: 局部去重
CREATE TEMP VIEW local_distinct AS
SELECT DISTINCT date, user_id FROM events;

-- Step 2: 全局计数
SELECT date, COUNT(user_id) FROM local_distinct GROUP BY date;
```

### 策略5：位图索引（低基数列）
```sql
-- 对于基数很低的列（如 status, category）
-- 某些数据库支持 bitmap index

-- PostgreSQL：创建部分索引
CREATE INDEX idx_active_users ON events (user_id) WHERE status = 'active';

-- 查询时利用索引
SELECT COUNT(DISTINCT user_id) FROM events WHERE status = 'active';
```

**性能对比示例：**

场景：1亿行数据，user_id 有 1000万不同值     

方法1：直接 COUNT(DISTINCT)
- 时间：~60秒
- 内存：需要存储 1000万个 ID

方法2：APPROX_COUNT_DISTINCT
- 时间：~5秒
- 内存：几 KB
- 误差：约 2%

方法3：两阶段聚合
- 时间：~20秒
- 减少了 shuffle 数据量


---



# Part 3: JOIN Operations

## Join Reorder Principle 重排序原理

- Join reorder optimization finds **the most efficient order** to join multiple tables. 
- The cost depends on **intermediate result sizes** 
  - joining in wrong order can create huge intermediate datasets. 
- Optimizers use statistics like row counts, distinct values, and histograms to estimate costs. 
- Common heuristics include: 
  - join smaller tables first, 
  - apply filters before joins, 
  - and put the most selective joins early. 
- Modern optimizers use **dynamic programming** or **greedy algorithms** to explore join orders. 
- In Spark, you can influence this with **join hints** or by **updating table statistics**. 
- Join order can mean the difference between seconds and hours.


**为什么 Join 顺序重要？**

```sql
-- 假设三张表
-- orders: 1亿行
-- customers: 100万行
-- regions: 100行

SELECT *
FROM orders o
JOIN customers c ON o.customer_id = c.id
JOIN regions r ON c.region_id = r.id;
```


顺序1：(orders ⋈ customers) ⋈ regions
- 第一步：1亿 × 100万 → 可能产生巨大中间结果
- 第二步：中间结果 × 100

顺序2：(customers ⋈ regions) ⋈ orders
- 第一步：100万 × 100 → 100万行（假设1对1）
- 第二步：100万 × 1亿 → 最终结果

顺序2 明显更好，中间结果小得多


**优化器如何决定顺序：**

### 1. 统计信息收集
```sql
-- PostgreSQL：分析表收集统计
ANALYZE orders;
ANALYZE customers;
ANALYZE regions;

-- Spark SQL：计算表统计
ANALYZE TABLE orders COMPUTE STATISTICS;
ANALYZE TABLE orders COMPUTE STATISTICS FOR COLUMNS customer_id;
```

### 2. 代价估算

代价因素：
- 表大小（行数、数据量）
- 列基数（distinct values）
- 选择性（过滤后保留比例）
- Join 类型（inner, left, cross）
- 可用索引


### 3. 搜索算法

动态规划（小表数量时）：
- 枚举所有可能的 join 顺序
- 计算每种顺序的代价
- 选择代价最小的

贪心算法（大表数量时）：
- 每步选择当前最优的 join
- 不保证全局最优，但速度快

Spark 使用的策略：
- 默认：基于统计的 CBO（Cost-Based Optimizer）
- 可以用 hints 手动指定


**手动干预 Join 顺序：**

```sql
-- Spark SQL hints
SELECT /*+ BROADCAST(r), MERGE(c, o) */
    *
FROM orders o
JOIN customers c ON o.customer_id = c.id
JOIN regions r ON c.region_id = r.id;

-- 指定 join 顺序
SELECT /*+ LEADING(r, c, o) */
    *
FROM orders o
JOIN customers c ON o.customer_id = c.id
JOIN regions r ON c.region_id = r.id;

-- PostgreSQL hints（需要 pg_hint_plan 扩展）
SELECT /*+ Leading((r c) o) */
    *
FROM ...
```

**最佳实践：**
```sql
-- 1. 先过滤后 Join（减少参与 Join 的数据量）
SELECT *
FROM (SELECT * FROM orders WHERE date > '2024-01-01') o  -- 先过滤
JOIN customers c ON o.customer_id = c.id;

-- 2. 小表放在 Join 的右边（某些数据库的 broadcast 策略）
SELECT *
FROM large_table l
JOIN small_table s ON l.key = s.key;

-- 3. 确保统计信息是最新的
ANALYZE TABLE orders COMPUTE STATISTICS;
```



## EXPLAIN Plan Analysis 
**EXPLAIN** shows how the database will execute your query. Key things to look for: 
- Scan types 
  - **sequential scan** means full table read, 
  - **index scan** is more efficient. 
- Join methods 
  - **nested loop** for small tables, 
  - **hash join** for equality conditions, 
  - **merge join** for sorted data. 
- Cost estimates - higher numbers mean more expensive. 
- Row estimates - large differences from actual indicate stale statistics. 
- In **Spark**, look for 
  - **Exchange** which means shuffle, 
  - **BroadcastHashJoin** which is efficient, 
  - and **data skew** in partition sizes. 
- Always compare estimated vs actual rows, and look for full table scans on large tables that should use indexes.


**基本用法：**
```sql
-- PostgreSQL
EXPLAIN SELECT * FROM orders WHERE status = 'pending';
EXPLAIN ANALYZE SELECT ...;  -- 实际执行并显示真实数据

-- MySQL
EXPLAIN SELECT ...;
EXPLAIN ANALYZE SELECT ...;  -- MySQL 8.0+

-- Spark SQL
df.explain()           -- 简单计划
df.explain(True)       -- 详细计划（所有阶段）
df.explain("formatted")  -- 格式化输出
```

**PostgreSQL EXPLAIN 解读：**
```sql
EXPLAIN ANALYZE 
SELECT c.name, COUNT(o.id)
FROM customers c
JOIN orders o ON c.id = o.customer_id
WHERE o.date > '2024-01-01'
GROUP BY c.name;

-- 输出示例：
HashAggregate  (cost=1234.56..1234.78 rows=100 width=40) (actual time=10.5..10.8 rows=95 loops=1)
  Group Key: c.name
  ->  Hash Join  (cost=100.00..1200.00 rows=5000 width=32) (actual time=2.1..8.5 rows=4800 loops=1)
        Hash Cond: (o.customer_id = c.id)
        ->  Seq Scan on orders o  (cost=0.00..800.00 rows=10000 width=16) (actual time=0.01..3.2 rows=9500 loops=1)
              Filter: (date > '2024-01-01')
              Rows Removed by Filter: 90500
        ->  Hash  (cost=50.00..50.00 rows=1000 width=24) (actual time=1.5..1.5 rows=1000 loops=1)
              ->  Seq Scan on customers c  (cost=0.00..50.00 rows=1000 width=24)
```

**关键指标解读：**

| 指标 | 含义 | 关注点 |
|------|------|--------|
| **cost** | 估算代价（启动代价..总代价）| 数值越大越慢 |
| **rows** | 估算行数 | **与 actual rows 差异大说明统计过时** |
| **actual time** | 实际执行时间（ms）| 找最耗时的节点 |
| **loops** | 执行次数 | >1 说明被外层多次调用 |
| **width** | 每行平均字节数 | 影响内存使用 |

### **扫描类型：**

- Seq Scan：全表扫描，最慢
- Index Scan：使用索引，较快
- Index Only Scan：只读索引，不访问表，最快
- Bitmap Index Scan：位图索引扫描，适合多条件


### **Join 类型：**

- Nested Loop：嵌套循环，适合小表或有索引
- Hash Join：哈希连接，适合等值 Join，需内存
- Merge Join：归并连接，适合已排序数据


### Spark SQL EXPLAIN 解读：
```sql
spark.sql("SELECT * FROM orders JOIN customers ON orders.cust_id = customers.id").explain(True)

-- 关注点：
== Physical Plan ==
*(2) BroadcastHashJoin [cust_id], [id], Inner, BuildRight
:- *(2) Filter isnotnull(cust_id)
:  +- *(2) FileScan parquet [id,cust_id,amount]
+- BroadcastExchange HashedRelationBroadcastMode
   +- *(1) Filter isnotnull(id)
      +- *(1) FileScan parquet [id,name]
```

**Spark 执行计划关键词：**

| 关键词 | 含义 | 性能影响 |
|--------|------|----------|
| **Exchange** | **Shuffle** 操作 | 昂贵，网络传输 |
| **BroadcastExchange** | 广播小表 | ✅ 好，避免 shuffle |
| **BroadcastHashJoin** | 广播 Hash Join | ✅ 好，高效 |
| **SortMergeJoin** | 排序归并 Join | 一般，需要排序 |
| **HashAggregate** | 哈希聚合 | ✅ 较好 |
| **Sort** | 排序操作 | 可能昂贵 |
| **Filter** | 过滤操作 | 检查是否下推 |
| **Project** | 列投影 | 检查是否裁剪 |

**优化检查清单：**

- 是否有不必要的 Seq Scan / 全表扫描？
- Join 类型是否合理？小表是否 broadcast？
- 估算行数与实际行数差异大吗？（需要 ANALYZE）
- 有没有多余的 Sort 操作？
- Filter 是否下推到了数据源？
- 只选择了需要的列吗？（Column Pruning）

---


# Part 4: Stored Procedures & Transactions 存储过程与事务

## Stored Procedures

- Stored procedures are **precompiled SQL programs stored in the database**.   
- Benefits include: 
  - reduced network traffic by executing logic on the server, 
  - reusable code across applications, 
  - and encapsulated business logic with controlled access. 
- They accept parameters, can include control flow like IF and LOOP, and can return result sets or output parameters. 
- Downsides are 
  - vendor lock-in since syntax varies, 
  - harder debugging, 
  - and potential for scattered business logic. 
- Use stored procedures for **complex data operations** that should stay close to the data, but avoid putting application logic there that belongs in your application layer.


**存储过程基本结构：**

```sql
-- PostgreSQL 语法
CREATE OR REPLACE PROCEDURE transfer_funds(
    sender_id INT,
    receiver_id INT,
    amount DECIMAL(10,2)
)
LANGUAGE plpgsql
AS $$
BEGIN
    -- 开始事务（存储过程内部）
    
    -- 扣减发送方余额
    UPDATE accounts SET balance = balance - amount WHERE id = sender_id;
    
    -- 检查余额是否足够
    IF NOT FOUND OR (SELECT balance FROM accounts WHERE id = sender_id) < 0 THEN
        RAISE EXCEPTION 'Insufficient funds';
    END IF;
    
    -- 增加接收方余额
    UPDATE accounts SET balance = balance + amount WHERE id = receiver_id;
    
    -- 记录交易日志
    INSERT INTO transaction_log (from_id, to_id, amount, created_at)
    VALUES (sender_id, receiver_id, amount, NOW());
    
    COMMIT;
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        RAISE;
END;
$$;

-- 调用存储过程
CALL transfer_funds(1, 2, 100.00);
```

**Function vs Procedure：**

| 特性 | Function | Procedure |
|------|----------|-----------|
| 返回值 | 必须返回值 | 可以无返回值 |
| 在 SQL 中使用 | 可以（SELECT func()）| 不可以 |
| 事务控制 | 不能 COMMIT/ROLLBACK（是表达式，不控制事务） | 可以 |
| 调用方式 | SELECT / 表达式中 | CALL 语句 |

**MySQL 存储过程：**
```sql
DELIMITER //
CREATE PROCEDURE GetEmployeesByDept(IN dept_name VARCHAR(50))
BEGIN
    SELECT * FROM employees WHERE department = dept_name;
END //
DELIMITER ;

-- 调用
CALL GetEmployeesByDept('Engineering');
```

**优缺点：**

优点：
- 减少网络往返（逻辑在数据库执行）
- 代码复用
- 安全性（可以限制直接表访问）
- 预编译，性能可能更好

缺点：
- 数据库厂商语法不同（移植性差）
- 调试困难
- 版本控制不便
- 业务逻辑分散
- 可能成为性能瓶颈




## Transactions | 事务

- Transactions ensure ACID properties: 
  - Atomicity - all operations succeed or all fail. 
  - Consistency - database moves from one valid state to another. 
  - Isolation - concurrent transactions don't interfere, controlled by isolation levels. 
  - Durability - committed changes survive failures. 
    
- **Isolation levels** from lowest to highest are: 
  - Read Uncommitted allows dirty reads, 
  - Read Committed prevents dirty reads, 
  - Repeatable Read prevents non-repeatable reads, and 
  - Serializable prevents phantom reads. 
- **Higher isolation means more consistency but less concurrency.** 
- Most applications use Read Committed as a balance. 
- Understanding these helps design correct concurrent data access."


**事务基本操作：**
```sql
-- 显式事务
BEGIN TRANSACTION;  -- 或 START TRANSACTION

UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

-- 检查是否有错误
-- 如果成功
COMMIT;
-- 如果失败
ROLLBACK;
```

**隔离级别详解：**

| 隔离级别 | dirty reads | non-repeatable reads | phantom reads 幻读 | Concurrency 并发性 |
|----------|------|------------|------|--------|
| Read Uncommitted | 可能 | 可能 | 可能 | 最高 |
| Read Committed | 不可能 | 可能 | 可能 | 高 |
| Repeatable Read | 不可能 | 不可能 | 可能 | 中 |
| Serializable | 不可能 | 不可能 | 不可能 | 最低 |

**三种读异常：**

1. 脏读（Dirty Read）：读到其他事务未提交的数据
- 事务A：UPDATE accounts SET balance = 500 WHERE id = 1;（未提交）
- 事务B：SELECT balance FROM accounts WHERE id = 1;  -- 读到 500
- 事务A：ROLLBACK;
- 结果：事务B读到了不存在的数据

2. 不可重复读（Non-repeatable Read）：同一事务内两次读取结果不同
- 事务A：SELECT balance FROM accounts WHERE id = 1;  -- 读到 100
- 事务B：UPDATE accounts SET balance = 200 WHERE id = 1; COMMIT;
- 事务A：SELECT balance FROM accounts WHERE id = 1;  -- 读到 200
- 结果：事务A两次读取结果不一致

3. 幻读（Phantom Read）：同一查询条件，两次查询行数不同
- 事务A：SELECT COUNT(*) FROM orders WHERE date = '2024-01-01';  -- 10行
- 事务B：INSERT INTO orders (date, ...) VALUES ('2024-01-01', ...); COMMIT;
- 事务A：SELECT COUNT(*) FROM orders WHERE date = '2024-01-01';  -- 11行
- 结果：出现了"幻影"行


**设置隔离级别：**
```sql
-- PostgreSQL
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;

-- MySQL
SET SESSION TRANSACTION ISOLATION LEVEL READ COMMITTED;

-- 查看当前隔离级别
SHOW TRANSACTION ISOLATION LEVEL;  -- PostgreSQL
SELECT @@transaction_isolation;     -- MySQL
```



## MERGE / UPDATE / UPSERT 

- `**MERGE**`, also called `UPSERT`, combines `INSERT` and `UPDATE` in one statement. 
- It matches source rows against target table - if match exists, update; if not, insert. 
- This is **atomic and more efficient** than separate INSERT and UPDATE statements. 
- Syntax varies: 
  - Standard SQL uses `MERGE`, 
  - PostgreSQL uses `INSERT ON CONFLICT`, 
  - MySQL uses `INSERT ON DUPLICATE KEY UPDATE`. 
- Use `MERGE` for 
  - syncing tables, 
  - loading data warehouses, or 
  - maintaining slowly changing dimensions. 
- It's essential for ETL processes where you need to update existing records and insert new ones in a single pass.

**标准 MERGE 语法（SQL Server, Oracle）：**
```sql
MERGE INTO target_table t
USING source_table s
ON t.id = s.id
WHEN MATCHED THEN
    UPDATE SET t.name = s.name, t.value = s.value, t.updated_at = CURRENT_TIMESTAMP
WHEN NOT MATCHED THEN
    INSERT (id, name, value, created_at)
    VALUES (s.id, s.name, s.value, CURRENT_TIMESTAMP);
```

**PostgreSQL UPSERT：**
```sql
-- INSERT ... ON CONFLICT
INSERT INTO products (id, name, price, updated_at)
VALUES (1, 'Widget', 10.00, NOW())
ON CONFLICT (id)  -- 冲突条件（通常是主键或唯一约束）
DO UPDATE SET
    name = EXCLUDED.name,
    price = EXCLUDED.price,
    updated_at = NOW();

-- EXCLUDED 指的是本次要插入的值
```

**MySQL UPSERT：**
```sql
-- INSERT ... ON DUPLICATE KEY UPDATE
INSERT INTO products (id, name, price)
VALUES (1, 'Widget', 10.00)
ON DUPLICATE KEY UPDATE
    name = VALUES(name),
    price = VALUES(price),
    updated_at = NOW();

-- 或使用 REPLACE（先删后插，会丢失未指定的列值）
REPLACE INTO products (id, name, price) VALUES (1, 'Widget', 10.00);
```

**Spark SQL MERGE（Delta Lake）：**
```sql
MERGE INTO target_table t
USING source_table s
ON t.id = s.id
WHEN MATCHED AND s.delete_flag = true THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

**使用场景：**

1. 数据仓库增量加载
2. 维度表 SCD（Slowly Changing Dimension）处理
3. 数据同步（从 OLTP 到 OLAP）
4. 去重插入



## Views | 视图

- Views are virtual tables defined by a query - they **don't store data themselves**. 
  - **Regular views** execute their query each time you access them. 
  - **Materialized views** store query results physically and must be refreshed - they trade storage for query speed. 
  - **Complex views** involve joins, aggregations, or subqueries and may not be updatable. 
- Use regular views for simplifying complex queries, implementing **row-level security**, and providing stable interfaces. 
- Use materialized views for expensive aggregations accessed frequently. 
- Key difference: regular views are always current, materialized views can be stale but faster.


**三种视图对比：**

| 特性 | 普通 View | Materialized View | Complex View |
|------|-----------|-------------------|--------------|
| 数据存储 | 不存储，每次查询执行 | 物理存储查询结果 | 不存储 |
| 查询速度 | 取决于底层查询 | 快（直接读存储的数据）| 可能慢 |
| 数据新鲜度 | 实时 | 需要刷新 | 实时 |
| 可更新性 | 简单视图可更新 | 通常不可直接更新 | 通常不可更新 |
| 使用场景 | 简化查询、权限控制 | 预计算聚合、报表加速 | 复杂业务逻辑封装 |

**普通 View：**
```sql
-- 创建视图
CREATE VIEW active_customers AS
SELECT id, name, email
FROM customers
WHERE status = 'active';

-- 使用视图（等同于执行定义的 SQL）
SELECT * FROM active_customers WHERE name LIKE 'A%';

-- 可更新视图条件（简单视图）：
-- - 不包含聚合函数
-- - 不包含 DISTINCT, GROUP BY, HAVING
-- - 不包含 UNION
-- - FROM 只有一个表
```

**Materialized View：**
```sql
-- PostgreSQL 创建物化视图
CREATE MATERIALIZED VIEW monthly_sales AS
SELECT 
    DATE_TRUNC('month', order_date) as month,
    product_id,
    SUM(amount) as total_sales,
    COUNT(*) as order_count
FROM orders
GROUP BY 1, 2;

-- 刷新物化视图
REFRESH MATERIALIZED VIEW monthly_sales;

-- 并发刷新（不阻塞查询，需要唯一索引）
REFRESH MATERIALIZED VIEW CONCURRENTLY monthly_sales;

-- 创建索引加速查询
CREATE INDEX idx_monthly_sales_product ON monthly_sales(product_id);
```

**Complex View 示例：**
```sql
-- 复杂视图：包含 JOIN 和聚合
CREATE VIEW customer_order_summary AS
SELECT 
    c.id,
    c.name,
    COUNT(o.id) as order_count,
    SUM(o.amount) as total_spent,
    MAX(o.order_date) as last_order_date
FROM customers c
LEFT JOIN orders o ON c.id = o.customer_id
GROUP BY c.id, c.name;

-- 这个视图不可更新（有聚合）
```

**视图的用途：**
```sql
-- 1. 简化复杂查询
CREATE VIEW order_details AS
SELECT o.*, c.name as customer_name, p.name as product_name
FROM orders o
JOIN customers c ON o.customer_id = c.id
JOIN products p ON o.product_id = p.id;

-- 2. 行级安全（用户只能看到自己的数据）
CREATE VIEW my_orders AS
SELECT * FROM orders WHERE user_id = current_user_id();

-- 3. 列级安全（隐藏敏感列）
CREATE VIEW public_customers AS
SELECT id, name, city FROM customers;  -- 不包含 email, phone

-- 4. 数据抽象（底层表结构变化时视图接口不变）
CREATE VIEW customer_info AS
SELECT id, first_name || ' ' || last_name as full_name FROM customers;
```

---



# Part 5: Partitioning & Indexing | 分区与索引

## Table Partitioning 

- Partitioning divides a large table into smaller, more manageable pieces while appearing as a single table. 
  - **Range partitioning** splits by value ranges like dates - ideal for time-series data. 
  - **List partitioning** uses discrete values like regions. 
  - **Hash partitioning** distributes data evenly by hash function. 
- Benefits include 
  - faster queries through partition pruning - only relevant partitions are scanned, 
  - easier maintenance like dropping old partitions, and 
  - parallel processing. 
- Design partitions based on query patterns - the partition key should appear in WHERE clauses frequently. 
- Too many small partitions or wrong key choice can hurt performance."


### **分区类型：**

#### 1. Range Partitioning（范围分区）
```sql
-- PostgreSQL
CREATE TABLE orders (
    id SERIAL,
    order_date DATE,
    amount DECIMAL(10,2)
) PARTITION BY RANGE (order_date);

-- 创建分区
CREATE TABLE orders_2023 PARTITION OF orders
    FOR VALUES FROM ('2023-01-01') TO ('2024-01-01');
CREATE TABLE orders_2024 PARTITION OF orders
    FOR VALUES FROM ('2024-01-01') TO ('2025-01-01');

-- 查询时自动分区裁剪
SELECT * FROM orders WHERE order_date = '2024-06-15';
-- 只扫描 orders_2024 分区
```

#### 2. List Partitioning（列表分区）
```sql
CREATE TABLE sales (
    id SERIAL,
    region VARCHAR(50),
    amount DECIMAL(10,2)
) PARTITION BY LIST (region);

CREATE TABLE sales_east PARTITION OF sales FOR VALUES IN ('NY', 'MA', 'CT');
CREATE TABLE sales_west PARTITION OF sales FOR VALUES IN ('CA', 'WA', 'OR');
CREATE TABLE sales_central PARTITION OF sales FOR VALUES IN ('TX', 'IL', 'OH');
```

#### 3. Hash Partitioning（哈希分区）
```sql
CREATE TABLE events (
    id SERIAL,
    user_id INT,
    event_type VARCHAR(50)
) PARTITION BY HASH (user_id);

CREATE TABLE events_p0 PARTITION OF events FOR VALUES WITH (MODULUS 4, REMAINDER 0);
CREATE TABLE events_p1 PARTITION OF events FOR VALUES WITH (MODULUS 4, REMAINDER 1);
CREATE TABLE events_p2 PARTITION OF events FOR VALUES WITH (MODULUS 4, REMAINDER 2);
CREATE TABLE events_p3 PARTITION OF events FOR VALUES WITH (MODULUS 4, REMAINDER 3);
```

### **分区裁剪（Partition Pruning）：**
```sql
-- 查询计划会显示只扫描相关分区
EXPLAIN SELECT * FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-03-31';

-- 输出类似：
-- Append
--   -> Seq Scan on orders_2024  -- 只扫描这个分区
--         Filter: (order_date >= '2024-01-01' AND order_date <= '2024-03-31')
```

### **分区维护：**
```sql
-- 删除旧分区（比 DELETE 快得多）
DROP TABLE orders_2022;

-- 添加新分区
CREATE TABLE orders_2025 PARTITION OF orders
    FOR VALUES FROM ('2025-01-01') TO ('2026-01-01');

-- 分离分区（用于归档）
ALTER TABLE orders DETACH PARTITION orders_2022;
```




## Index Principles (B+ Tree) 

- B+ tree is the most common index structure. 
- It's a balanced tree where **all data pointers are in leaf nodes**, and leaf nodes are linked for range scans. 
- Non-leaf nodes contain only **keys for navigation**. 
- Benefits: 
  - consistent O(log n) lookups, 
  - efficient range queries by following leaf links, and 
  - remains balanced through splits and merges. 
- Index selection matters: use indexes for columns in WHERE, JOIN, and ORDER BY clauses. 
- Covering indexes include all needed columns, avoiding table lookups. 
- But indexes have costs - they slow down writes and use storage. 
- The optimizer decides whether using an index is faster than a full scan.


**B+ Tree 结构：**
```text
                    [30 | 60]                    ← 根节点（只存 key）
                   /    |    \
          [10|20]    [40|50]    [70|80]          ← 中间节点（只存 key）
          /  |  \     / | \     /  |  \
        [1-9][10-19][20-29]...[70-79][80-89]     ← 叶子节点（存 key + 数据指针）
                ↔ 双向链表连接 ↔

特点：
1. 所有叶子节点在同一层（平衡树）
2. 叶子节点包含所有 key 和数据指针
3. 叶子节点通过链表连接（方便范围扫描）
4. 非叶子节点只存索引 key（更多分支，树更矮）
```

**为什么用 B+ Tree？**

1. 查找复杂度 O(log n)
   - 1000万行数据，大约只需 3-4 次磁盘 I/O
   
2. 范围查询高效
   - 叶子节点链表连接，顺序扫描

3. 插入删除保持平衡
   - 通过分裂和合并维护平衡

4. 磁盘友好
   - 节点大小通常等于磁盘页大小（4KB/8KB）
   - 减少 I/O 次数


### **索引类型：**
```sql
-- 1. B-Tree 索引（默认，最常用）
CREATE INDEX idx_name ON users(name);

-- 2. 唯一索引
CREATE UNIQUE INDEX idx_email ON users(email);

-- 3. 复合索引（多列）
CREATE INDEX idx_name_age ON users(name, age);
-- 遵循最左前缀原则：可用于 (name), (name, age)，不能只用于 (age)

-- 4. 覆盖索引（包含查询所需的所有列）
CREATE INDEX idx_covering ON orders(customer_id) INCLUDE (order_date, amount);
-- 查询可以只读索引，不需要回表

-- 5. 部分索引（只索引部分行）
CREATE INDEX idx_active ON users(email) WHERE status = 'active';

-- 6. 哈希索引（只支持等值查询）
CREATE INDEX idx_hash ON users USING HASH (id);
```

### **索引使用判断：**
```sql
-- 什么时候会用索引？
-- 1. WHERE 子句中的等值/范围条件
SELECT * FROM users WHERE email = 'test@example.com';  -- 用索引

-- 2. JOIN 条件
SELECT * FROM orders o JOIN users u ON o.user_id = u.id;  -- 用索引

-- 3. ORDER BY（如果索引顺序匹配）
SELECT * FROM users ORDER BY name;  -- 可能用索引

-- 什么时候不用索引？
-- 1. 小表（全表扫描更快）
-- 2. 选择性低的列（如性别）
-- 3. 函数包裹列
SELECT * FROM users WHERE LOWER(name) = 'john';  -- 不用索引
-- 解决：函数索引
CREATE INDEX idx_lower_name ON users(LOWER(name));

-- 4. 不等于条件
SELECT * FROM users WHERE status != 'deleted';  -- 可能不用索引

-- 5. LIKE 前缀通配符
SELECT * FROM users WHERE name LIKE '%john';  -- 不用索引
SELECT * FROM users WHERE name LIKE 'john%';  -- 用索引
```

### **索引代价：**

1. 存储空间：索引占用额外磁盘空间
2. 写入开销：INSERT/UPDATE/DELETE 需要维护索引
3. 选择开销：优化器需要决定是否使用索引

最佳实践：
- 只在必要的列上创建索引
- 高选择性列优先（distinct 值多）
- 定期分析索引使用情况
- 删除未使用的索引

---



# Part 6: Query Optimization 


## General Query Optimization 

- Query optimization follows several principles: 
  - First, select only needed columns - avoid SELECT star. 
  - Second, filter early - push WHERE conditions as close to data source as possible. 
  - Third, use appropriate indexes - ensure indexed columns are in WHERE and JOIN clauses. 
  - Fourth, avoid functions on indexed columns - they prevent index usage. 
  - Fifth, optimize JOINs - use proper join types, consider join order, broadcast small tables. 
  - Sixth, limit result sets - use LIMIT when you don't need all rows. 
  - Seventh, use EXPLAIN to verify the execution plan. 
  - Always measure before and after optimization to confirm improvement.


**优化检查清单：**

### 1. SELECT 优化
```sql
-- ❌ 避免 SELECT *
SELECT * FROM large_table;

-- ✅ 只选需要的列
SELECT id, name, email FROM large_table;
```

### 2. WHERE 优化
```sql
-- ❌ 函数包裹导致索引失效
SELECT * FROM orders WHERE YEAR(order_date) = 2024;

-- ✅ 改写为范围条件
SELECT * FROM orders 
WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01';

-- ❌ 隐式类型转换
SELECT * FROM users WHERE phone = 1234567890;  -- phone 是 VARCHAR

-- ✅ 类型匹配
SELECT * FROM users WHERE phone = '1234567890';
```

### 3. JOIN 优化
```sql
-- ❌ 笛卡尔积（缺少 JOIN 条件）
SELECT * FROM orders, customers;

-- ✅ 明确 JOIN 条件
SELECT * FROM orders o JOIN customers c ON o.customer_id = c.id;

-- ✅ 小表放在 JOIN 右边（某些数据库的优化）
SELECT * FROM large_table l JOIN small_table s ON l.key = s.key;
```

### 4. 子查询优化
```sql
-- ❌ 相关子查询（每行执行一次）
SELECT * FROM orders o
WHERE customer_id IN (SELECT id FROM customers c WHERE c.region = o.region);

-- ✅ 改写为 JOIN
SELECT o.* FROM orders o
JOIN customers c ON o.customer_id = c.id AND o.region = c.region;
```

### 5. 聚合优化
```sql
-- ❌ 先聚合后过滤
SELECT department, AVG(salary) FROM employees
GROUP BY department
HAVING department IN ('Sales', 'Engineering');

-- ✅ 先过滤后聚合
SELECT department, AVG(salary) FROM employees
WHERE department IN ('Sales', 'Engineering')
GROUP BY department;
```

### 6. 分页优化
```sql
-- ❌ 深分页（OFFSET 越大越慢）
SELECT * FROM orders ORDER BY id LIMIT 10 OFFSET 1000000;

-- ✅ 基于游标的分页
SELECT * FROM orders WHERE id > 1000000 ORDER BY id LIMIT 10;

-- ✅ 延迟关联（先查 ID，再查详情）
SELECT * FROM orders o
JOIN (SELECT id FROM orders ORDER BY id LIMIT 10 OFFSET 1000000) t
ON o.id = t.id;
```

### 7. 批量操作优化
```sql
-- ❌ 逐条插入
INSERT INTO logs VALUES (1, 'a');
INSERT INTO logs VALUES (2, 'b');
...

-- ✅ 批量插入
INSERT INTO logs VALUES (1, 'a'), (2, 'b'), (3, 'c'), ...;

-- ✅ 批量更新（用 CASE）
UPDATE products SET price = CASE id
    WHEN 1 THEN 10.00
    WHEN 2 THEN 20.00
    WHEN 3 THEN 30.00
END
WHERE id IN (1, 2, 3);
```

<img src='./sql_cheatsheets/query_optimization.png' width=850>



## CTE vs Subquery | CTE vs 子查询

- CTEs, Common Table Expressions, use `WITH` clause to define named temporary result sets. 
- Compared to subqueries: 
  - CTEs are more readable, especially for complex queries, and 
  - can be referenced multiple times in the main query. 
- CTEs can be recursive for hierarchical data. 
- Performance-wise, some databases materialize CTEs while others inline them like subqueries - PostgreSQL materializes by default. Subqueries are fine for simple cases and are sometimes optimized better. 
- Use CTEs for readability, reuse within a query, and recursive operations. 
- Use subqueries for simple one-off filtering. 
- In Spark, CTEs often get inlined for optimization."

**基本语法对比：**

```sql
-- CTE (Common Table Expression)
WITH active_customers AS (
    SELECT * FROM customers WHERE status = 'active'
),
recent_orders AS (
    SELECT * FROM orders WHERE order_date > CURRENT_DATE - INTERVAL '30 days'
)
SELECT c.name, COUNT(o.id) as order_count
FROM active_customers c
JOIN recent_orders o ON c.id = o.customer_id
GROUP BY c.name;

-- 等价的子查询写法
SELECT c.name, COUNT(o.id) as order_count
FROM (SELECT * FROM customers WHERE status = 'active') c
JOIN (SELECT * FROM orders WHERE order_date > CURRENT_DATE - INTERVAL '30 days') o
ON c.id = o.customer_id
GROUP BY c.name;
```

**对比分析：**

| 特性 | CTE | Subquery |
|------|-----|----------|
| **可读性** | 好，命名清晰 | 嵌套深时难读 |
| **复用** | 同一查询中可多次引用 | 每次使用需重写 |
| **递归** | 支持 | 不支持 |
| **物化** | 某些数据库会物化 | 通常内联优化 |
| **调试** | 易于单独运行测试 | 需要拆开测试 |

**递归 CTE（处理层级数据）：**
```sql
-- 组织架构树：找出某员工的所有下属
WITH RECURSIVE subordinates AS (
    -- 基础情况：直接下属
    SELECT id, name, manager_id, 1 as level
    FROM employees
    WHERE manager_id = 100  -- 从 ID=100 的经理开始
    
    UNION ALL
    
    -- 递归情况：下属的下属
    SELECT e.id, e.name, e.manager_id, s.level + 1
    FROM employees e
    JOIN subordinates s ON e.manager_id = s.id
    WHERE s.level < 10  -- 防止无限递归
)
SELECT * FROM subordinates;
```

**性能考虑：**
```sql
-- PostgreSQL：CTE 默认物化（可能影响优化）
-- 强制内联（PostgreSQL 12+）
WITH active_customers AS MATERIALIZED (
    SELECT * FROM customers WHERE status = 'active'
)
-- 或
WITH active_customers AS NOT MATERIALIZED (
    SELECT * FROM customers WHERE status = 'active'
)

-- Spark SQL：CTE 通常被内联优化
-- 如果需要强制物化，可以 cache
spark.sql("WITH cte AS (SELECT ...) SELECT * FROM cte").cache()
```

**使用建议：**

使用 CTE 当：
- 查询复杂，需要提高可读性
- 同一个中间结果需要多次使用
- 需要递归查询
- 团队规范要求

使用 Subquery 当：
- 简单的单次过滤
- 性能关键且确认内联更优
- EXISTS / NOT EXISTS 条件

---



# Part 7: NoSQL & Distributed Databases



## NoSQL Database Types | NoSQL 数据库类型

- NoSQL databases are non-relational, designed for specific use cases. 
- Four main types: 
  - Key-Value stores like Redis - simple, fast, good for caching and sessions. 
  - Document stores like MongoDB - flexible JSON documents, good for content management and catalogs. 
  - Column-family stores like Cassandra - wide columns, excellent for time-series and high write throughput. 
  - Graph databases like Neo4j - relationships as first-class citizens, ideal for social networks and recommendations. 
- Choose based on data model fit, query patterns, and scale requirements. 
- NoSQL trades some ACID guarantees for horizontal scalability and flexibility.


| 类型 | 代表产品 | 数据模型 | 适用场景 |
|------|----------|----------|----------|
| **Key-Value** | Redis, DynamoDB | key → value | 缓存、会话、简单查询 |
| **Document** | MongoDB, CouchDB | JSON 文档 | CMS、产品目录、用户档案 |
| **Column-Family** | Cassandra, HBase | 宽列存储 | 时序数据、日志、IoT |
| **Graph** | Neo4j, Neptune | 节点+关系 | 社交网络、推荐、欺诈检测 |

**各类型示例：**

```python
# Key-Value (Redis)
redis_client.set("user:123", json.dumps({"name": "Alice", "age": 25}))
user = json.loads(redis_client.get("user:123"))

# Document (MongoDB)
db.users.insert_one({
    "_id": "user123",
    "name": "Alice",
    "addresses": [
        {"type": "home", "city": "NYC"},
        {"type": "work", "city": "Boston"}
    ]
})
db.users.find({"addresses.city": "NYC"})

# Column-Family (Cassandra)
# Table: users
# Row Key: user_id
# Columns: name, email, login_history (可以有数千列)

# Graph (Neo4j - Cypher)
# CREATE (alice:Person {name: 'Alice'})
# CREATE (bob:Person {name: 'Bob'})
# CREATE (alice)-[:FRIENDS_WITH]->(bob)
# MATCH (p:Person)-[:FRIENDS_WITH*2]-(friend) WHERE p.name = 'Alice' RETURN friend
```

**SQL vs NoSQL 选择：**

选择 SQL 当：
- 数据结构清晰且稳定
- 需要复杂查询和 JOIN
- 需要 ACID 事务
- 数据一致性是首要需求

选择 NoSQL 当：
- 数据结构灵活多变
- 需要水平扩展
- 高吞吐量写入
- 可以接受最终一致性





## CAP Theorem

- CAP theorem states distributed systems can only guarantee two of three properties: 
  - Consistency - all nodes see the same data simultaneously, 
  - Availability - every request gets a response, 
  - Partition tolerance - system operates despite network failures. 
- Since network partitions are inevitable, the real choice is between consistency and availability during failures. 
- CP systems like HBase choose consistency - may reject requests during partitions. 
- AP systems like Cassandra choose availability - may return stale data. 
- Most modern systems let you tune this tradeoff. 
- Understanding CAP helps design appropriate data architectures for your requirements.


**CAP 分类：**
```text
┌─────────────────────────────────────────────────────┐
│                     CAP Theorem                     │
│                                                     │
│           C                                         │
│          /  \                                       │
│         /    \                                      │
│        /  CP  \         CA = 单机系统                │
│       /        \        CP = HBase, MongoDB         │
│      /    CA    \       AP = Cassandra, DynamoDB    │
│     A ────────── P                                  │
│           AP                                        │
└─────────────────────────────────────────────────────┘

实际选择：
- 网络分区不可避免，所以必须选 P
- 真正的选择是 C vs A
- 大多数系统允许配置一致性级别
```

**一致性级别（以 Cassandra 为例）：**

- 写入一致性
  - ONE: 写入一个节点即返回（最快，可能丢数据）
  - QUORUM: 写入多数节点（平衡）
  - ALL: 写入所有节点（最慢，最一致）
- 读取一致性
  - ONE: 读取一个节点
  - QUORUM: 读取多数节点，返回最新值
  - ALL: 读取所有节点
- 强一致性公式：W + R > N
  - W=写入节点数, R=读取节点数, N=总副本数
  - 例：3副本，W=2, R=2 → 2+2>3 → 强一致




## ACID vs BASE 

- ACID and BASE represent **different consistency models**. 
- ACID - Atomicity, Consistency, Isolation, Durability 
  - guarantees strong consistency, 
  - used in traditional relational databases. 
  - Transactions are all-or-nothing, isolated from each other. 
- BASE 
  - Basically Available, Soft state, Eventually consistent
  - accepts temporary inconsistency for availability and performance. 
  - Used in many NoSQL systems. 
  - Eventually consistent means all replicas converge to the same state given time without new updates. 
- Choose ACID for financial transactions, inventory management. 
- Choose BASE for social media feeds, analytics, where slight delay in consistency is acceptable.


| 特性 | ACID | BASE |
|------|------|------|
| **一致性** | 强一致性 | 最终一致性 |
| **可用性** | 可能牺牲可用性 | 优先可用性 |
| **使用场景** | 金融、库存、订单 | 社交媒体、分析、日志 |
| **代表系统** | PostgreSQL, MySQL | Cassandra, DynamoDB |
| **性能** | 相对较低 | 高吞吐 |
| **扩展性** | 垂直扩展为主 | 水平扩展 |

**Eventually consistent 最终一致性的含义：**

时间线：
- T0: 写入节点A（data=100）
- T1: 节点B读取（可能返回旧值）
- T2: 复制到节点B
- T3: 节点B读取（返回100）

"最终"的时间通常是毫秒到秒级    
对于很多应用场景是可以接受的


## Database Normalization | 数据库规范化

- Normalization organizes data to reduce redundancy and improve integrity. 
- First Normal Form: eliminate repeating groups, ensure atomic values. 
- Second Normal Form: meet 1NF plus remove partial dependencies - every non-key column depends on the entire primary key. 
- Third Normal Form: meet 2NF plus remove transitive dependencies - non-key columns depend only on the primary key, not on other non-key columns. 
- BCNF is stricter - every determinant is a candidate key. 
- For analytics, we often denormalize for query performance. 
- The key is knowing when to normalize for integrity versus denormalize for speed.


**规范化范式：**

```text
1NF（第一范式）：
- 每列都是原子值（不可再分）
- 消除重复组

❌ 违反 1NF:
| id | phones              |
| 1  | 123-456, 789-012    |  ← 不是原子值

✓ 符合 1NF:
| id | phone     |
| 1  | 123-456   |
| 1  | 789-012   |

---

2NF（第二范式）：
- 满足 1NF
- 消除部分依赖（非主键列完全依赖于主键）

❌ 违反 2NF:
| order_id | product_id | product_name | quantity |
| 1        | A          | Widget       | 10       |
# product_name 只依赖 product_id，不依赖完整主键

✓ 符合 2NF:
Orders: | order_id | product_id | quantity |
Products: | product_id | product_name |

---

3NF（第三范式）：
- 满足 2NF
- 消除传递依赖（非主键列不依赖其他非主键列）

❌ 违反 3NF:
| employee_id | department_id | department_name |
# department_name 依赖 department_id，而非 employee_id

✓ 符合 3NF:
Employees: | employee_id | department_id |
Departments: | department_id | department_name |
```

**OLTP vs OLAP 规范化策略：**

OLTP（事务处理）：   
- 高度规范化（3NF 或更高）
- 减少数据冗余
- 保证数据一致性
- 写入优化

OLAP（分析处理）：     
- 反规范化（Star Schema）
- 允许冗余换取查询速度
- 减少 JOIN 操作
- 读取优化



## Sharding & Replication 

- Sharding **horizontally partitions data** across multiple database instances - each shard holds a subset of rows. 
- Sharding strategies: 
  - **hash-base**d distributes evenly but makes range queries hard, 
  - **range-based** keeps related data together but can cause hotspots. 
- Replication creates copies of data across nodes for fault tolerance and read scaling. 
  - **Master-slave replication** has one write node and multiple read replicas.
  - **Multi-master** allows writes to any node but requires conflict resolution. 
  - Combine both: shard for write scaling and storage, replicate each shard for read scaling and availability. 
- Key challenge is choosing the right shard key.


**Sharding（分片）：**
```text
                    ┌─────────────┐
                    │   Router    │
                    └──────┬──────┘
                           │
           ┌───────────────┼───────────────┐
           ↓               ↓               ↓
     ┌──────────┐    ┌──────────┐    ┌──────────┐
     │ Shard 1  │    │ Shard 2  │    │ Shard 3  │
     │ A-H      │    │ I-P      │    │ Q-Z      │
     └──────────┘    └──────────┘    └──────────┘

分片策略：
1. Hash Sharding: shard = hash(key) % num_shards
   + 数据均匀分布
   - 范围查询需要查所有分片

2. Range Sharding: shard = 根据 key 范围
   + 范围查询高效
   - 可能产生热点

3. Directory Sharding: 查表确定分片
   + 灵活
   - 需要额外的查找服务
```

**Replication（复制）：**
```text
主从复制（Master-Slave）：
┌──────────┐
│  Master  │ ← 所有写入
└────┬─────┘
     │ 复制
┌────┴────────────────┐
↓         ↓           ↓
┌─────┐  ┌─────┐  ┌─────┐
│Slave│  │Slave│  │Slave│ ← 读取分散
└─────┘  └─────┘  └─────┘

多主复制（Multi-Master）：
┌──────────┐    ┌──────────┐
│ Master 1 │←──→│ Master 2 │
└──────────┘    └──────────┘
     ↑               ↑
     写入           写入

需要处理写入冲突：
- Last-Write-Wins
- 向量时钟
- 应用层解决
```

**选择分片键的考虑：**

好的分片键：
- ✓ 高基数（很多不同值）
- ✓ 查询经常使用
- ✓ 分布均匀
- ✓ 不会频繁变化

例子：
- user_id（用户数据）
- order_id（订单数据）
- 组合键（tenant_id + user_id）

避免：
- ✗ 时间戳（造成热点）
- ✗ 低基数（如 status）
- ✗ 频繁更新的字段


---



# Part 8: Advanced SQL Topics 



## DELETE vs TRUNCATE vs DROP | 删除操作对比

- DELETE 
  - removes rows based on condition, 
  - is logged, 
  - can be rolled back, 
  - and fires triggers. 
  - Slow for large tables but precise. 
- TRUNCATE 
  - removes all rows, 
  - is minimally logged, 
  - cannot be rolled back in most databases, 
  - doesn't fire triggers, 
  - and resets identity columns. 
  - Much faster for clearing tables. 
- DROP removes the entire table structure and data permanently. 
- Use DELETE when you need WHERE conditions or need to rollback. 
- Use TRUNCATE to quickly clear a table you'll reuse. 
- Use DROP when removing the table entirely. 
- In production, prefer DELETE for safety despite being slower.


| 特性 | DELETE | TRUNCATE | DROP |
|------|--------|----------|------|
| **作用** | 删除指定行 | 删除所有行 | 删除整个表 |
| **WHERE** | 支持 | 不支持 | N/A |
| **事务日志** | 完整记录 | 最小记录 | 最小记录 |
| **回滚** | 可以 | 大多数数据库不可 | 不可 |
| **触发器** | 触发 | 不触发 | N/A |
| **自增列** | 不重置 | 重置 | N/A |
| **速度** | 慢 | 快 | 快 |
| **锁** | 行锁 | 表锁 | 表锁 |

```sql
-- DELETE: 逐行删除，可以回滚
DELETE FROM orders WHERE status = 'cancelled';

-- TRUNCATE: 快速清空，不可回滚
TRUNCATE TABLE temp_orders;

-- DROP: 删除表结构
DROP TABLE IF EXISTS old_orders;

-- 安全的生产实践
BEGIN TRANSACTION;
DELETE FROM orders WHERE order_date < '2020-01-01';
-- 检查影响的行数
-- COMMIT 或 ROLLBACK
```



## Locks & Concurrency | 锁与并发

- Database locks **prevent concurrent transactions from conflicting**. 
- **Shared locks** allow multiple readers but block writers. 
- **Exclusive locks** block all other access. 
- **Row-level locks** affect only specific rows - better concurrency. 
- **Table-level locks** are simpler but block more operations. 
- **Deadlocks** occur when transactions wait for each other's locks - databases detect and kill one transaction. 
- To minimize locking issues: keep transactions short, access tables in consistent order, use appropriate isolation levels, and consider optimistic locking with version columns for read-heavy workloads. 
- Understanding locking is crucial for debugging production performance issues.


**锁的类型：**

共享锁（Shared Lock / S Lock）：
- 读取时获取
- 多个事务可以同时持有
- 阻止写入

排他锁（Exclusive Lock / X Lock）：
- 写入时获取
- 只有一个事务可以持有
- 阻止读取和写入


锁兼容矩阵：  
```text
       | S Lock | X Lock |
-------|--------|--------|
S Lock |   ✓    |   ✗    |
X Lock |   ✗    |   ✗    |
```

**锁粒度：**

表锁（Table Lock）：
- 简单，开销小
- 并发度低
- 适合批量操作

行锁（Row Lock）：
- 高并发
- 开销大，可能升级为表锁
- 适合 OLTP

页锁（Page Lock）：
- 介于两者之间
- 某些数据库使用


**死锁示例：**
```sql
-- 事务1
BEGIN;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;  -- 锁住 id=1
-- 自动锁： 数据库自动给该行加 row-level exclusive lock
-- 等待 id=2 的锁...
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

-- 事务2（同时执行）
BEGIN;
UPDATE accounts SET balance = balance - 50 WHERE id = 2;   -- 锁住 id=2
-- 等待 id=1 的锁...
UPDATE accounts SET balance = balance + 50 WHERE id = 1;

-- 结果：死锁！数据库会检测并回滚其中一个事务
```

**乐观锁 vs 悲观锁：**
```sql
-- 悲观锁：假设会有冲突，提前锁定
SELECT * FROM products WHERE id = 1 FOR UPDATE;
-- 执行业务逻辑
UPDATE products SET stock = stock - 1 WHERE id = 1;
COMMIT;

-- 乐观锁：假设没有冲突，更新时检查
-- 使用版本号
UPDATE products 
SET stock = stock - 1, version = version + 1 
WHERE id = 1 AND version = 5;
-- 如果 affected_rows = 0，说明有冲突，需要重试
```

### 不同数据库差异
| 数据库| 锁机制| 
| ------| ----| 
| PostgreSQL| MVCC + 行锁| 
| MySQL InnoDB| MVCC + gap lock| 
| Oracle| 行级锁| 
| SQL Server| 行 / 页 / 表锁| 

MVCC = Multi-Version Concurrency Control（多版本并发控制）
- 不通过“读锁”阻塞读，而是给数据保留多个版本。
- 所以，读不会阻塞写，写不会阻塞读

在现代数据工程里：
- OLAP 很少手动加锁
- Spark 不用数据库锁
- 事务控制主要在 OLTP

如果你做：
- 库存系统
- 金融系统
- 任务调度

才会涉及显式锁。




## Query Performance Tuning | 查询性能调优

- Query tuning is systematic: 
- First, identify slow queries using **slow query log or monitoring tools**. 
- Second, analyze execution plans with **EXPLAIN** - look for full table scans, missing indexes, bad join order. 
- Third, **optimize**: add missing indexes, rewrite queries to be sargable, reduce data scanned with better filters, consider denormalization. 
- Fourth, verify improvement and monitor. 
- Common wins: ensure indexed columns aren't wrapped in functions, use covering indexes, avoid SELECT star, push filters early. 
- For complex queries, consider materialized views or pre-aggregation. 
- Always test with production-like data volumes.


**调优流程：**
```text
1. 识别慢查询
   ├── 慢查询日志
   ├── APM 工具（DataDog, New Relic）
   └── pg_stat_statements (PostgreSQL)

2. 分析执行计划
   ├── EXPLAIN ANALYZE
   ├── 查看扫描类型
   ├── 检查 JOIN 方法
   └── 比较估算 vs 实际行数

3. 优化
   ├── 添加/调整索引
   ├── 重写查询
   ├── 调整数据库配置
   └── 考虑架构变更

4. 验证和监控
   ├── 比较前后性能
   ├── 持续监控
   └── 记录变更
```

**常见优化技巧：**
```sql
-- 1. 使索引可用（Sargable）
-- ❌ 索引失效
WHERE YEAR(created_at) = 2024
-- ✓ 索引可用
WHERE created_at >= '2024-01-01' AND created_at < '2025-01-01'

-- 2. 避免 SELECT *
-- ❌ 
SELECT * FROM orders WHERE customer_id = 123
-- ✓ 只选需要的列
SELECT order_id, amount, status FROM orders WHERE customer_id = 123

-- 3. 使用覆盖索引
CREATE INDEX idx_orders_customer ON orders(customer_id) INCLUDE (amount, status);

-- 4. 优化 IN 子查询
-- ❌ 可能性能差
SELECT * FROM orders WHERE customer_id IN (SELECT id FROM vip_customers)
-- ✓ 改用 JOIN
SELECT o.* FROM orders o JOIN vip_customers v ON o.customer_id = v.id

-- 5. 避免 OR，使用 UNION
-- ❌ 可能不用索引
SELECT * FROM users WHERE email = 'a@b.com' OR phone = '123456'
-- ✓ 可以用两个索引
SELECT * FROM users WHERE email = 'a@b.com'
UNION
SELECT * FROM users WHERE phone = '123456'

-- 6. 批量操作
-- ❌ 逐条插入
INSERT INTO logs VALUES (1, 'a');
INSERT INTO logs VALUES (2, 'b');
-- ✓ 批量插入
INSERT INTO logs VALUES (1, 'a'), (2, 'b'), (3, 'c');
```

---



## Common SQL Patterns | 常用 SQL 模式

- Several SQL patterns solve common problems efficiently. 
- Running totals use window functions with unbounded preceding. 
- Finding gaps in sequences uses self-join or window functions comparing adjacent rows. 
- Deduplication keeps one row per group using ROW_NUMBER and filtering to rank 1. 
- Pivot transforms rows to columns using CASE expressions or PIVOT operator. 
- Unpivot does the reverse. 
- Recursive CTEs traverse hierarchies like org charts. 
- Islands and gaps problems identify consecutive groups. 
- Knowing these patterns accelerates problem-solving and leads to efficient queries. Most complex analytical questions can be solved by combining these fundamental patterns.

**常用模式示例：**

```sql
-- 1. 累计求和（Running Total）
SELECT 
    date,
    amount,
    SUM(amount) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING) as running_total
FROM sales;

-- 2. 找出缺失的数字（Gaps）
WITH numbered AS (
    SELECT id, id - ROW_NUMBER() OVER (ORDER BY id) as grp
    FROM existing_ids
)
SELECT MIN(id) as gap_start, MAX(id) as gap_end
FROM numbered
GROUP BY grp;

-- 3. 去重保留一条（Deduplication）
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY email ORDER BY created_at DESC) as rn
    FROM users
)
SELECT * FROM ranked WHERE rn = 1;

-- 4. 行转列（Pivot）
SELECT 
    product_id,
    SUM(CASE WHEN month = 'Jan' THEN sales ELSE 0 END) as jan_sales,
    SUM(CASE WHEN month = 'Feb' THEN sales ELSE 0 END) as feb_sales,
    SUM(CASE WHEN month = 'Mar' THEN sales ELSE 0 END) as mar_sales
FROM monthly_sales
GROUP BY product_id;

-- 5. 递归查询（Hierarchy）
WITH RECURSIVE org_tree AS (
    -- 基础：顶层员工
    SELECT id, name, manager_id, 1 as level
    FROM employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- 递归：下属
    SELECT e.id, e.name, e.manager_id, t.level + 1
    FROM employees e
    JOIN org_tree t ON e.manager_id = t.id
)
SELECT * FROM org_tree ORDER BY level, name;

-- 6. 连续区间（Islands）
WITH numbered AS (
    SELECT 
        date,
        date - INTERVAL '1 day' * ROW_NUMBER() OVER (ORDER BY date) as island_id
    FROM login_dates
)
SELECT 
    MIN(date) as start_date,
    MAX(date) as end_date,
    COUNT(*) as consecutive_days
FROM numbered
GROUP BY island_id;

-- 7. 同比环比（Period Comparison）
SELECT 
    current.month,
    current.sales,
    previous.sales as prev_month_sales,
    last_year.sales as last_year_sales,
    (current.sales - previous.sales) / previous.sales * 100 as mom_growth,
    (current.sales - last_year.sales) / last_year.sales * 100 as yoy_growth
FROM monthly_sales current
LEFT JOIN monthly_sales previous 
    ON current.month = previous.month + INTERVAL '1 month'
LEFT JOIN monthly_sales last_year 
    ON current.month = last_year.month + INTERVAL '1 year';
```

---



# Part 9: [Interview Questions by Level](session-18-BigData/0_2_sql_interview_questions.ipynb)


## Junior SQL Questions | 初级 SQL 问题


基础查询：
- SELECT, WHERE, ORDER BY, LIMIT 的用法
- 如何去重？DISTINCT vs GROUP BY
- NULL 如何处理？
- LIKE 和通配符的使用
- 日期函数的基本使用

JOIN 操作：
- INNER JOIN vs LEFT JOIN vs RIGHT JOIN
- 如何写一个多表 JOIN？
- 自连接（Self Join）是什么？

聚合函数：
- COUNT, SUM, AVG, MAX, MIN 的区别
- GROUP BY 和 HAVING 的区别
- 如何找出重复记录？


## Mid-Level SQL Questions | 中级 SQL 问题


窗口函数：
- ROW_NUMBER vs RANK vs DENSE_RANK
- 如何计算累计求和？
- 如何计算移动平均？
- LAG 和 LEAD 的使用

子查询和 CTE：
- 相关子查询 vs 非相关子查询
- CTE 的好处是什么？
- 递归 CTE 如何工作？

性能优化：
- 如何分析 EXPLAIN 计划？
- 索引什么时候失效？
- 如何优化慢查询？


## Senior SQL Questions | 高级 SQL 问题


高级优化：
- 覆盖索引的原理
- 查询重写技巧
- 分区表的设计
- 物化视图的使用场景

事务与并发：
- 四种隔离级别的区别
- 死锁如何产生和解决？
- 乐观锁 vs 悲观锁

架构设计：
- 规范化 vs 反规范化的权衡
- 分片策略的选择
- 读写分离的实现


# Quick Reference


## SQL Function Categories | SQL 函数分类

```sql
-- ============ 窗口函数 ============
-- 排名函数
ROW_NUMBER() OVER (PARTITION BY col ORDER BY col2)
RANK() OVER (ORDER BY col)
DENSE_RANK() OVER (ORDER BY col)
NTILE(n) OVER (ORDER BY col)

-- 偏移函数
LAG(col, n, default) OVER (ORDER BY col)
LEAD(col, n, default) OVER (ORDER BY col)
FIRST_VALUE(col) OVER (PARTITION BY col ORDER BY col2)
LAST_VALUE(col) OVER (PARTITION BY col ORDER BY col2 
                       ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)

-- 窗口聚合
SUM(col) OVER (PARTITION BY col ORDER BY col2 
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  -- 累计和
AVG(col) OVER (ORDER BY col ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)  -- 移动平均

-- ============ 聚合函数 ============
COUNT(*), COUNT(col), COUNT(DISTINCT col)
SUM(col), AVG(col)
MAX(col), MIN(col)
STRING_AGG(col, ',')  -- PostgreSQL
GROUP_CONCAT(col)     -- MySQL
ARRAY_AGG(col)        -- 聚合成数组

-- ============ 字符串函数 ============
CONCAT(str1, str2)
SUBSTRING(str, start, length)
UPPER(str), LOWER(str)
TRIM(str), LTRIM(str), RTRIM(str)
REPLACE(str, from, to)
SPLIT_PART(str, delimiter, position)  -- PostgreSQL
LENGTH(str), CHAR_LENGTH(str)

-- ============ 日期函数 ============
CURRENT_DATE, CURRENT_TIMESTAMP
DATE_TRUNC('month', date)  -- PostgreSQL
EXTRACT(YEAR FROM date)
DATE_ADD(date, INTERVAL 1 DAY)  -- MySQL
date + INTERVAL '1 day'         -- PostgreSQL
DATEDIFF(date1, date2)          -- MySQL
```




## SQL Optimization Cheat Sheet | SQL 优化速查表

```text
┌────────────────────────────────────────────────────────────────────┐
│                    SQL OPTIMIZATION CHEAT SHEET                    │
├────────────────────────────────────────────────────────────────────┤
│ INDEXING PRINCIPLES:                                               │
│   • B+ Tree: O(log n) lookup, efficient range scans                │
│   • Index columns in: WHERE, JOIN, ORDER BY                        │
│   • Composite index: follows leftmost prefix rule                  │
│   • Covering index: includes all needed columns                    │
├────────────────────────────────────────────────────────────────────┤
│ QUERY PATTERNS:                                                    │
│   ✗ SELECT * → ✓ SELECT needed_columns                             │
│   ✗ WHERE FUNC(col) = val → ✓ WHERE col = INVERSE_FUNC(val)        │
│   ✗ LIKE '%value' → ✓ LIKE 'value%'                                │
│   ✗ Large OFFSET → ✓ Cursor-based pagination                       │
├────────────────────────────────────────────────────────────────────┤
│ JOIN OPTIMIZATION:                                                 │
│   • Small table on build side (right side for hash join)           │
│   • Ensure join columns have indexes                               │
│   • Consider broadcast for small dimension tables                  │
│   • Filter before join when possible                               │
├────────────────────────────────────────────────────────────────────┤
│ AGGREGATION:                                                       │
│   • COUNT(DISTINCT): consider APPROX_COUNT_DISTINCT                │
│   • Filter with WHERE before GROUP BY, not HAVING                  │
│   • Pre-aggregate in subquery for complex counts                   │
├────────────────────────────────────────────────────────────────────┤
│ WINDOW FUNCTIONS:                                                  │
│   ROW_NUMBER: unique sequence (1,2,3,4)                            │
│   RANK: ties share, then skip (1,1,3,4)                            │
│   DENSE_RANK: ties share, no skip (1,1,2,3)                        │
│   Frame: ROWS = physical, RANGE = logical                          │
├────────────────────────────────────────────────────────────────────┤
│ EXPLAIN PLAN RED FLAGS:                                            │
│   ⚠ Seq Scan on large table (missing index?)                       │
│   ⚠ Nested Loop with large tables (need hash/merge join?)          │
│   ⚠ Sort before Join (can avoid with index?)                       │
│   ⚠ Large row estimate mismatch (stale statistics?)                │
└────────────────────────────────────────────────────────────────────┘
```




## Transaction Isolation Quick Reference | 事务隔离级别速查

```text
┌─────────────────────────────────────────────────────────────────────┐
│                    ISOLATION LEVELS                                 │
├──────────────────┬────────────┬───────────────┬─────────────────────┤
│ Level            │ Dirty Read │ Non-Repeatable│ Phantom Read        │
├──────────────────┼────────────┼───────────────┼─────────────────────┤
│ Read Uncommitted │ Possible   │ Possible      │ Possible            │
│ Read Committed   │ Prevented  │ Possible      │ Possible            │
│ Repeatable Read  │ Prevented  │ Prevented     │ Possible            │
│ Serializable     │ Prevented  │ Prevented     │ Prevented           │
├──────────────────┴────────────┴───────────────┴─────────────────────┤
│ Default: PostgreSQL = Read Committed                                │
│          MySQL InnoDB = Repeatable Read                             │
│          SQL Server = Read Committed                                │
└─────────────────────────────────────────────────────────────────────┘

Dirty Read: Reading uncommitted changes from another transaction
Non-Repeatable Read: Same query returns different values
Phantom Read: Same query returns different row count
```




## Index Type Summary | 索引类型总结

```text
┌─────────────────────────────────────────────────────────────────────┐
│                       INDEX TYPES                                   │
├─────────────────┬───────────────────────────────────────────────────┤
│ B-Tree          │ Default, all comparisons, range queries           │
├─────────────────┼───────────────────────────────────────────────────┤
│ Hash            │ Equality only (=), very fast point lookups        │
├─────────────────┼───────────────────────────────────────────────────┤
│ GIN             │ Full-text search, array contains, JSONB           │
├─────────────────┼───────────────────────────────────────────────────┤
│ GiST            │ Geometric, range types, full-text                 │
├─────────────────┼───────────────────────────────────────────────────┤
│ BRIN            │ Large tables with natural ordering (time-series)  │
├─────────────────┼───────────────────────────────────────────────────┤
│ Bitmap          │ Low cardinality columns, data warehouses          │
└─────────────────┴───────────────────────────────────────────────────┘

Composite Index Rule: (A, B, C) can be used for:
  ✓ WHERE A = ?
  ✓ WHERE A = ? AND B = ?
  ✓ WHERE A = ? AND B = ? AND C = ?
  ✗ WHERE B = ?  (must start from leftmost)
  ✗ WHERE A = ? AND C = ?  (can only use A part)
```


